<a href="https://colab.research.google.com/github/Nosindiso13/yield-app/blob/main/cropyield_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np

In [3]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input, decode_predictions
import numpy as np

# Load a pre-trained MobileNetV2 model for demonstration
pest_model = MobileNetV2(weights='imagenet')

def detect_pest(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)

    preds = pest_model.predict(x)
    return decode_predictions(preds, top=3)[0]

print("Pest detection model (MobileNetV2) loaded successfully.")

14536120/14536120 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Pest detection model (MobileNetV2) loaded successfully.


In [4]:
import requests

# Streamlit public URL obtained from previous ngrok execution
streamlit_public_url = "https://libidinally-unconcordant-jose.ngrok-free.dev"

print(f"Checking Streamlit app at: {streamlit_public_url}")

try:
    response = requests.get(streamlit_public_url, timeout=10)
    response.raise_for_status() # Raise an exception for HTTP errors (4xx or 5xx)

    print(f"\nStreamlit app is running and reachable! Status Code: {response.status_code}")
    # Optionally, print first few lines of content to confirm it's an HTML page
    print("First 500 characters of response content:")
    print(response.text[:500])

except requests.exceptions.ConnectionError:
    print(f"Error: Could not connect to the Streamlit app at {streamlit_public_url}. The app might not be running or the ngrok tunnel is down.")
except requests.exceptions.Timeout:
    print(f"Error: Request to {streamlit_public_url} timed out. The Streamlit app might be slow to respond or not fully started.")
except requests.exceptions.RequestException as e:
    print(f"An HTTP error occurred: {e}")
    if response is not None:
        print(f"Response status code: {response.status_code}")
        print(f"Response body (first 500 chars): {response.text[:500]}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Checking Streamlit app at: https://libidinally-unconcordant-jose.ngrok-free.dev
An HTTP error occurred: 404 Client Error: Not Found for url: https://libidinally-unconcordant-jose.ngrok-free.dev/
Response status code: 404
Response body (first 500 chars): <!DOCTYPE html>
<html class="h-full" lang="en-US" dir="ltr">
  <head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Regular-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-RegularItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
  


In [5]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')
import os

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import cross_val_score, train_test_split, RepeatedKFold # Added RepeatedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# __ Optional XGBoost ──────────────────────────────────────────────────────────
try:
    from xgboost import XGBRegressor
    USE_XGB = True
except ImportError:
    USE_XGB = False

# __ 1. Load & inspect data ────────────────────────────────────────────────────
print("=" * 60)
print("CROP YIELD PREDICTION — ML PIPELINE")
print("=" * 60)

# --- FIX: Removed direct download due to persistent 404 errors ---
# Please upload 'yield_df.csv' directly to your Colab environment (e.g., via the files tab)
# If you don't have the file, you can search for 'yield_df.csv' online or on Kaggle.

# Load the dataset from local file
try:
    df = pd.read_csv('yield_df.csv', index_col=0)
except FileNotFoundError:
    print("Error: 'yield_df.csv' not found. Please upload the file to your Colab environment.")
    raise

if df.empty:
    print("Error: DataFrame 'df' is empty after reading CSV. The uploaded file might be malformed.")
    raise ValueError("Exiting due to empty DataFrame, cannot proceed with analysis.")
# --- End of fix for data loading ---

print(f"\nDataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nTarget (hg/ha_yield) stats:")
print(df['hg/ha_yield'].describe().round(1))
# __ 2. Feature engineering ────────────────────────────────────────────────────
# Log-transform target to handle skew
df['log_yield'] = np.log1p(df['hg/ha_yield'])

# Rename for convenience
df = df.rename(columns={
    'year': 'Year',
    'Rainfall': 'rainfall',
    'Pesticides': 'pesticides',
    'avg_temp': 'temperature'
})

CATEGORICAL = ['Area', 'Item']
NUMERICAL   = ['Year', 'rainfall', 'pesticides', 'temperature']
TARGET      = 'log_yield'

X = df[CATEGORICAL + NUMERICAL]
y = df[TARGET]

# __ 3. Preprocessing pipeline ─────────────────────────────────────────────────
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL),
    ('num', StandardScaler(), NUMERICAL),
])

# __ 4. Define models ──────────────────────────────────────────────────────────
models = {
    'Random Forest':        RandomForestRegressor(n_estimators=200, max_depth=15,
                                                  min_samples_leaf=2, n_jobs=-1,
                                                  random_state=42),
    'Gradient Boosting':    GradientBoostingRegressor(n_estimators=200, max_depth=5,
                                                      learning_rate=0.1,
                                                      subsample=0.8, random_state=42),
    'Ridge Regression':     Ridge(alpha=10.0),
    'KNN Regressor':        KNeighborsRegressor(n_neighbors=10, weights='distance',
                                                n_jobs=-1),
}

if USE_XGB:
    models['XGBoost'] = XGBRegressor(n_estimators=300, max_depth=6,
                                     learning_rate=0.05, subsample=0.8,
                                     colsample_bytree=0.8, random_state=42,
                                     tree_method='hist')

# __ 5. Train / test split ──────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"\nTrain size: {len(X_train):,}  |  Test size: {len(X_test):,}")

# __ 6. Train, evaluate, cross-validate ────────────────────────────────────────
print("\n" + "=" * 60)
print("MODEL EVALUATION (log-scale RMSE, R², MAE + 5-fold CV R²)")
print("=" * 60)

results = {}

# Define RepeatedKFold for more robust cross-validation
rkf = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)

for name, model in models.items():
    pipe = Pipeline([('prep', preprocessor), ('model', model)])

    # Fit
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    # Metrics on log scale
    rmse_log = np.sqrt(mean_squared_error(y_test, y_pred))
    r2        = r2_score(y_test, y_pred)
    mae_log   = mean_absolute_error(y_test, y_pred)

    # Back-transform for interpretable MAE (hg/ha)
    mae_orig = mean_absolute_error(np.expm1(y_test), np.expm1(y_pred))

    # 5-fold, 3-repeat cross-validation R²
    cv_scores = cross_val_score(pipe, X, y, cv=rkf, scoring='r2', n_jobs=-1)

    results[name] = {
        'pipe':     pipe,
        'y_pred':   y_pred,
        'rmse_log': rmse_log,
        'r2':       r2,
        'mae_log':  mae_log,
        'mae_orig': mae_orig,
        'cv_mean':  cv_scores.mean(),
        'cv_std':   cv_scores.std(),
    }

    print(f"\n{name}")
    print(f"  RMSE (log):       {rmse_log:.4f}")
    print(f"  R²:               {r2:.4f}")
    print(f"  MAE (hg/ha):      {mae_orig:,.0f}")
    print(f"  CV R² (5-fold, 3-repeat):   {cv_scores.mean():.4f} \u00b1 {cv_scores.std():.4f}")

# Best model
best_name = max(results, key=lambda k: results[k]['cv_mean'])
print(f"\n\u2605  Best model by CV R²: {best_name} ({results[best_name]['cv_mean']:.4f})")

# __ 7. Feature importance (Random Forest) ─────────────────────────────────────
rf_pipe  = results['Random Forest']['pipe']
rf_model = rf_pipe.named_steps['model']
ohe_cols = rf_pipe.named_steps['prep'].transformers_[0][1].get_feature_names_out(CATEGORICAL)
feature_names = list(ohe_cols) + NUMERICAL

importances = pd.Series(rf_model.feature_importances_, index=feature_names)

# Aggregate OHE columns back to Area / Item
def agg_importance(imp, prefix):
    mask = imp.index.str.startswith(prefix)
    return imp[mask].sum()

agg = pd.Series({
    'Area':        agg_importance(importances, 'Area'),
    'Item (crop)': agg_importance(importances, 'Item'),
    'Year':        importances['Year'],
    'Rainfall':    importances['rainfall'],
    'Pesticides':  importances['pesticides'],
    'Temperature': importances['temperature'],
}).sort_values(ascending=True)

# __ 8. Plots ───────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 12))
fig.patch.set_facecolor('#F8F7F4')
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

PALETTE = {
    'Random Forest':     '#1D9E75',
    'Gradient Boosting': '#534AB7',
    'Ridge Regression':  '#185FA5',
    'KNN Regressor':     '#BA7517',
    'XGBoost':           '#D85A30',
}

# __ 8a. CV R² comparison bar chart ────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
names  = list(results.keys())
cv_means = [results[n]['cv_mean'] for n in names]
cv_stds  = [results[n]['cv_std']  for n in names]
colors   = [PALETTE.get(n, '#888') for n in names]

bars = ax1.barh(names, cv_means, xerr=cv_stds, color=colors, height=0.55,
                error_kw=dict(elinewidth=1.2, capsize=4, ecolor='#555'))
ax1.set_xlabel('CV R² score', fontsize=10)
ax1.set_title('Cross-validated R²\n(5-fold, 3-repeat, higher = better)', fontsize=11, fontweight='500') # Updated title
ax1.set_xlim(0, 1.05)
ax1.axvline(0.9, color='#aaa', lw=0.8, ls='--', alpha=0.6)
for bar, val in zip(bars, cv_means):
    ax1.text(val + 0.01, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=9)
ax1.set_facecolor('#F8F7F4')
ax1.spines[['top','right']].set_visible(False)

# __ 8b. RMSE comparison ────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
rmses = [results[n]['rmse_log'] for n in names]
bars2 = ax2.barh(names, rmses, color=colors, height=0.55)
ax2.set_xlabel('RMSE (log scale)', fontsize=10)
ax2.set_title('RMSE on test set\n(lower = better)', fontsize=11, fontweight='500')
for bar, val in zip(bars2, rmses):
    ax2.text(val + 0.002, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=9)
ax2.set_facecolor('#F8F7F4')
ax2.spines[['top','right']].set_visible(False)

# __ 8c. Feature importance (RF) ───────────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
feat_colors = ['#1D9E75' if v > 0.15 else '#9FE1CB' for v in agg.values]
ax3.barh(agg.index, agg.values, color=feat_colors, height=0.55)
ax3.set_xlabel('Importance', fontsize=10)
ax3.set_title('Feature importance\n(Random Forest, aggregated)', fontsize=11, fontweight='500')
ax3.set_facecolor('#F8F7F4')
ax3.spines[['top','right']].set_visible(False)

# __ 8d–f. Actual vs predicted (top 3 models by CV R²) ─────────────────────────
top3 = sorted(results, key=lambda k: results[k]['cv_mean'], reverse=True)[:3]

for i, name in enumerate(top3):
    ax = fig.add_subplot(gs[1, i])
    y_t  = np.expm1(y_test)
    y_p  = np.expm1(results[name]['y_pred'])
    r2v  = results[name]['r2']

    ax.scatter(y_t, y_p, alpha=0.15, s=6, color=PALETTE.get(name, '#888'),
               rasterized=True)
    lim = max(y_t.max(), y_p.max()) * 1.05
    ax.plot([0, lim], [0, lim], 'k--', lw=1, alpha=0.5, label='Perfect fit')
    ax.set_xlabel('Actual yield (hg/ha)', fontsize=9)
    ax.set_ylabel('Predicted yield (hg/ha)', fontsize=9)
    ax.set_title(f'{name}\nR² = {r2v:.4f}', fontsize=10, fontweight='500')
    ax.set_facecolor('#F8F7F4')
    ax.spines[['top','right']].set_visible(False)
    ax.ticklabel_format(style='sci', axis='both', scilimits=(0,0))

fig.suptitle('Crop Yield Prediction — Model Evaluation', fontsize=14,
             fontweight='600', y=1.01)

# Create the directory if it doesn't exist
os.makedirs('/mnt/user-data/outputs/', exist_ok=True)
plt.savefig('/mnt/user-data/outputs/crop_yield_model_evaluation.png',
            dpi=150, bbox_inches='tight', facecolor='#F8F7F4')
print("\nPlot saved.")

# __ 9. Summary table ───────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("SUMMARY TABLE")
print("=" * 60)
summary = pd.DataFrame({
    'Model':       names,
    'CV R²':       [f"{results[n]['cv_mean']:.4f} \u00b1 {results[n]['cv_std']:.4f}" for n in names],
    'Test R²':     [f"{results[n]['r2']:.4f}"       for n in names],
    'RMSE (log)':  [f"{results[n]['rmse_log']:.4f}" for n in names],
    'MAE (hg/ha)': [f"{results[n]['mae_orig']:,.0f}" for n in names],
}).set_index('Model')
print(summary.to_string())

print(f"\n\u2605  Best model: {best_name}")
print("\nDone.")

CROP YIELD PREDICTION — ML PIPELINE

Dataset shape: (370, 7)
Columns: ['Area', 'Item', 'year', 'hg/ha_yield', 'Rainfall', 'Pesticides', 'avg_temp']

Target (hg/ha_yield) stats:
count       370.0
mean      42066.8
std       34563.0
min        2976.0
25%       16781.2
50%       45515.8
75%       49092.3
max      172628.0
Name: hg/ha_yield, dtype: float64

Train size: 296  |  Test size: 74

MODEL EVALUATION (log-scale RMSE, R², MAE + 5-fold CV R²)

Random Forest
  RMSE (log):       0.2520
  R²:               0.9200
  MAE (hg/ha):      3,870
  CV R² (5-fold, 3-repeat):   0.9123 ± 0.0336

Gradient Boosting
  RMSE (log):       0.2244
  R²:               0.9366
  MAE (hg/ha):      3,329
  CV R² (5-fold, 3-repeat):   0.9285 ± 0.0246

Ridge Regression
  RMSE (log):       0.5615
  R²:               0.6029
  MAE (hg/ha):      20,556
  CV R² (5-fold, 3-repeat):   0.6376 ± 0.0508

KNN Regressor
  RMSE (log):       0.5405
  R²:               0.6320
  MAE (hg/ha):      13,691
  CV R² (5-fold, 3-repea

In [6]:
print('Installing authentication-related libraries...')
!pip install passlib[bcrypt] python-jose[cryptography]
print('Installation complete.')

Installing authentication-related libraries...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.8/150.8 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 525.6/525.6 kB 21.1 MB/s eta 0:00:00
Installation complete.


In [7]:
%%writefile main.py
from pydantic import BaseModel, Field
from typing import List, Dict
import pandas as pd
import numpy as np
import joblib
import os
import io
from PIL import Image
from datetime import datetime, timedelta

# FastAPI imports
from fastapi import FastAPI, HTTPException, Depends, status, UploadFile, File
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm

# Authentication imports
from passlib.context import CryptContext
from jose import JWTError, jwt

# Import for Gemini API
import google.generativeai as genai

# New Tensorflow imports for pest detection
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image

# --- 0. Configure Gemini & Security ---
# In a separate process, we use environment variables instead of userdata.get()
GOOGLE_API_KEY = os.getenv('YIELD_API_KEY')
if GOOGLE_API_KEY:
    genai.configure(api_key=GOOGLE_API_KEY)

# Security settings
SECRET_KEY = os.getenv("SECRET_KEY", "a_very_secret_key_that_should_be_changed_in_production")
ALGORITHM = "HS256"
ACCESS_TOKEN_EXPIRE_MINUTES = 30

pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="token") # This tokenUrl points to our /login endpoint

# --- 1. Pydantic Models ---

# User models
class User(BaseModel):
    username: str
    email: str | None = None
    full_name: str | None = None
    disabled: bool | None = None
    hashed_password: str # This should ideally not be exposed directly in responses

class UserInDB(User):
    pass

class Token(BaseModel:
    access_token: str
    token_type: str

class TokenData(BaseModel:
    username: str | None = None

class CropYieldRequest(BaseModel):
    Area: str
    Item: str
    Year: int = Field(..., gt=0)
    rainfall: float = Field(..., ge=0.0)
    pesticides: float = Field(..., ge=0.0)
    temperature: float = Field(..., ge=-50.0, le=70.0)

class ChatRequest(BaseModel):
    message: str

# --- 2. FastAPI App Instance ---
app = FastAPI(title="Crop Yield & Farmer Advisory API")

# --- 3. In-memory 'Database' & User Management Functions ---
# For demonstration, a simple dictionary to store users
fake_users_db = {}

# Helper functions for password hashing
def verify_password(plain_password, hashed_password):
    return pwd_context.verify(plain_password, hashed_password)

def get_password_hash(password):
    return pwd_context.hash(password)

# Helper functions for JWT tokens
def create_access_token(data: dict, expires_delta: timedelta | None = None):
    to_encode = data.copy()
    if expires_delta:
        expire = datetime.utcnow() + expires_delta
    else:
        expire = datetime.utcnow() + timedelta(minutes=ACCESS_TOKEN_EXPIRE_MINUTES)
    to_encode.update({"exp": expire})
    encoded_jwt = jwt.encode(to_encode, SECRET_KEY, algorithm=ALGORITHM)
    return encoded_jwt

async def get_current_user(token: str = Depends(oauth2_scheme)):
    credentials_exception = HTTPException(
        status_code=status.HTTP_401_UNAUTHORIZED,
        detail="Could not validate credentials",
        headers={"WWW-Authenticate": "Bearer"},
    )
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        username: str = payload.get("sub")
        if username is None:
            raise credentials_exception
        token_data = TokenData(username=username)
    except JWTError:
        raise credentials_exception
    # In a real app, you would fetch the user from a database here
    # For now, we'll just check if the username exists in our fake_users_db (after registration)
    if token_data.username not in fake_users_db:
        raise credentials_exception
    return fake_users_db[token_data.username]

# --- 4. Global Models ---
model_pipeline = None
gemini_model = None
pest_model = None
MODEL_PATH = 'model_artifacts/xgboost_pipeline.joblib'

@app.on_event('startup')
async def load_models():
    global model_pipeline, gemini_model, pest_model
    if not os.path.exists(MODEL_PATH):
        # Handle missing model more gracefully, maybe raise a warning but don't stop startup
        print(f"Warning: Model file not found at {MODEL_PATH}. Prediction endpoint might not work.")
        model_pipeline = None # Set to None if not found
    else:
        try:
            model_pipeline = joblib.load(MODEL_PATH)
        except Exception as e:
            print(f"Error loading crop yield model: {e}")
            model_pipeline = None

    if GOOGLE_API_KEY:
        try:
            gemini_model = genai.GenerativeModel('gemini-pro')
        except Exception as e:
            print(f"Error loading Gemini model: {e}")
            gemini_model = None
    else:
        print("Warning: YIELD_API_KEY not set. Chat endpoint might not work.")
        gemini_model = None

    try:
        pest_model = MobileNetV2(weights='imagenet')
    except Exception as e:
        print(f"Error loading MobileNetV2 model: {e}")
        pest_model = None

    print("All models attempted to load.")

# --- 5. API Endpoints ---

@app.get('/')
async def read_root():
    return {'status': 'active', 'message': 'API is running'}

@app.post('/predict')
async def predict(request_data: List[CropYieldRequest], current_user: User = Depends(get_current_user)): # Secured endpoint
    if model_pipeline is None: raise HTTPException(status_code=500, detail="Model not loaded")
    df = pd.DataFrame([item.model_dump() for item in request_data])
    log_predictions = model_pipeline.predict(df)
    return {'predictions': np.expm1(log_predictions).tolist()}

@app.post('/chat')
async def chat(request: ChatRequest, current_user: User = Depends(get_current_user)): # Secured endpoint
    if not gemini_model:
        raise HTTPException(status_code=500, detail="Gemini API not configured")
    response = await gemini_model.generate_content_async(request.message)
    return {'response': response.text}

@app.post('/detect_pest')
async def detect_pest_endpoint(file: UploadFile = File(...), current_user: User = Depends(get_current_user)): # Secured endpoint
    if pest_model is None: raise HTTPException(status_code=500, detail="Pest detection model not loaded")
    image_bytes = await file.read()
    img = Image.open(io.BytesIO(image_bytes)).resize((224, 224))
    x = preprocess_input(np.expand_dims(image.img_to_array(img), axis=0))
    preds = decode_predictions(pest_model.predict(x), top=3)[0]
    return {"detections": [{"label": l, "description": d, "probability": float(p)} for (_, l, p) in preds]}


Writing main.py


In [8]:
import pandas as pd

try:
    df_check = pd.read_csv('yield_df.csv', index_col=0)
    print("Successfully loaded 'yield_df.csv'.")
    print("\nFirst 5 rows of the DataFrame:")
    display(df_check.head())
    print("\nDataFrame Information (column types and non-null counts):")
    df_check.info()
    print("\nDescriptive Statistics:")
    display(df_check.describe())
except FileNotFoundError:
    print("Error: 'yield_df.csv' not found. Please ensure the file is uploaded to your Colab environment.")
except Exception as e:
    print(f"An error occurred while reading the CSV: {e}")

Successfully loaded 'yield_df.csv'.

First 5 rows of the DataFrame:


,Area,Item,year,hg/ha_yield,Rainfall,Pesticides,avg_temp
0,,,,,,,
1.0,Zambia,Cassava,2022.0,62040.0,1020.0,1080.0,21.43
2.0,Zambia,Maize,1990.0,14316.0,1020.0,1080.0,21.43
3.0,Zambia,Potatoes,1990.0,90000.0,1020.0,1080.0,21.43
4.0,Zambia,"Rice, paddy",1990.0,9664.0,1020.0,1080.0,21.43
5.0,Zambia,Sorghum,1990.0,4042.0,1020.0,1080.0,21.43



DataFrame Information (column types and non-null counts):
<class 'pandas.core.frame.DataFrame'>
Index: 370 entries, 1.0 to nan
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Area         370 non-null    object 
 1   Item         370 non-null    object 
 2   year         370 non-null    float64
 3   hg/ha_yield  370 non-null    float64
 4   Rainfall     370 non-null    float64
 5   Pesticides   370 non-null    float64
 6   avg_temp     370 non-null    float64
dtypes: float64(5), object(2)
memory usage: 23.1+ KB

Descriptive Statistics:


,year,hg/ha_yield,Rainfall,Pesticides,avg_temp
count,370.000000,370.000000,370.000000,370.000000,370.000000
mean,2000.156973,42066.771829,816.827240,2497.046408,20.941152
std,6.035271,34562.980133,205.846450,1266.282456,0.418628
min,1990.000000,2976.000000,493.765755,716.000000,20.140000
25%,1995.000000,16781.250000,655.296497,1670.000000,20.660000
50%,2001.356250,45515.846395,666.240315,1670.000000,20.790000
75%,2002.573391,49092.301562,1020.000000,3439.454309,21.190000
max,2022.000000,172628.000000,1020.000000,6753.000000,21.970000


In [9]:
import joblib
import os

print("Joblib and OS modules imported.")

Joblib and OS modules imported.


In [10]:
import os
import joblib

# Ensure the exact directory expected by main.py exists
output_dir = 'model_artifacts'
os.makedirs(output_dir, exist_ok=True)

# Save the XGBoost pipeline to the correct location
model_path = os.path.join(output_dir, 'xgboost_pipeline.joblib')
try:
    # Assuming 'results' is available from cell bF695J1hEgBB
    joblib.dump(results['XGBoost']['pipe'], model_path)
    print(f"Model verified and saved to: {model_path}")
except NameError:
    print("Error: 'results' dictionary not found. Please ensure the training cell (bF695J1hEgBB) has been executed.")

Model verified and saved to: model_artifacts/xgboost_pipeline.joblib


In [11]:
print('''import streamlit as st
import requests # Will be removed later, kept for initial modification clarity
import json
import pandas as pd
import numpy as np
import pickle # Replaced joblib with pickle
import os
import io
from PIL import Image
import asyncio # Added for async operations

# Imports for Gemini API
import google.genai as genai # Corrected import
from google.colab import userdata

# Imports for Pest Detection
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image

# --- Configuration & Global Variables ---
MODEL_PATH = 'model_artifacts/xgboost_pipeline.joblib'

# --- Caching Models for Efficiency ---
@st.cache_resource
def load_crop_yield_model():
    if not os.path.exists(MODEL_PATH):
        st.error(f"Crop Yield Model file not found at {MODEL_PATH}.")
        return None
    try:
        with open(MODEL_PATH, 'rb') as f:
            pipeline = pickle.load(f) # Replaced joblib.load with pickle.load
        return pipeline
    except Exception as e:
        st.error(f"Error loading Crop Yield model: {e}")
        return None

@st.cache_resource
def load_pest_detection_model():
    try:
        model = MobileNetV2(weights='imagenet')
        return model
    except Exception as e:
        st.error(f"Error loading Pest Detection model: {e}")
        return None

@st.cache_resource
def load_gemini_model():
    try:
        # Ensure API key is configured
        api_key = userdata.get('crop_key')
        if not api_key:
            st.error("Google API Key not found. Please set 'crop_key' in Colab secrets.")
            return None
        # Pass API key directly to the GenerativeModel constructor
        model = genai.GenerativeModel('gemini-pro', api_key=api_key)
        return model
    except Exception as e:
        st.error(f"Error initializing Gemini model: {e}")
        return None

# --- Helper Functions (moved from main.py) ---
def predict_yield_helper(input_data_df: pd.DataFrame, model_pipeline_local) -> list[float]:
    if model_pipeline_local is None:
        st.error("Crop Yield Model is not loaded.")
        return []

    # Ensure column order matches training data
    expected_columns = ['Area', 'Item', 'Year', 'rainfall', 'pesticides', 'temperature']
    df_processed = input_data_df[expected_columns]

    log_predictions = model_pipeline_local.predict(df_processed)
    original_scale_predictions = np.expm1(log_predictions)
    return original_scale_predictions.tolist()

def detect_pest_helper(image_bytes: bytes, pest_model_local) -> list[dict]:
    if pest_model_local is None:
        st.error("Pest Detection Model is not loaded.")
        return []
    try:
        img = Image.open(io.BytesIO(image_bytes)).resize((224, 224))
        x = image.img_to_array(img)
        x = np.expand_dims(x, axis=0)
        x = preprocess_input(x)

        preds = pest_model_local.predict(x)
        decoded_preds = decode_predictions(preds, top=3)[0]

        results = [{"label": label, "description": description, "probability": float(prob)}
                   for (_, label, prob) in decoded_preds]
        return results
    except Exception as e:
        st.error(f"Image processing or pest detection failed: {e}")
        return []

async def get_gemini_response(message: str, gemini_model_local) -> str:
    if gemini_model_local is None:
        st.error("Gemini model is not initialized.")
        return "Error: Gemini model not available."
    try:
        response = await gemini_model_local.generate_content_async(message)
        return response.text
    except Exception as e:
        st.error(f"Gemini API call failed: {e}")
        return "Error: Could not get a response from AI."

# --- Load all models at startup ---
crop_yield_pipeline = load_crop_yield_model()
pest_detection_model = load_pest_detection_model()
gemini_llm = load_gemini_model()

# --- Streamlit App Layout ---
st.set_page_config(page_title="Crop Yield Predictor & Advisory", layout="centered")

st.title("Crop Yield Prediction & Farmer Advisory")
st.markdown("Use the sections below to predict crop yields, detect pests, and get AI-powered advice.")

# Create tabs
tab1, tab2, tab3 = st.tabs(["Crop Yield Prediction", "Pest Detection", "Farmer Advisory"])

with tab1:
    st.subheader("Input Parameters for Yield Prediction")

    AREAS = ['Zambia', 'Zimbabwe']
    ITEMS = ['Wheat', 'Maize', 'Potatoes', 'Rice, paddy', 'Sorghum', 'Soybeans', 'Cassava', 'Sweet potatoes']
    MIN_YEAR = 2020
    MAX_YEAR = 2030
    MIN_RAINFALL = 0.0
    MAX_RAINFALL = 3500.0
    MIN_PESTICIDES = 0.0
    MAX_PESTICIDES = 150000.0
    MIN_TEMP = -20.0
    MAX_TEMP = 40.0

    with st.form("prediction_form"):
        col1, col2 = st.columns(2)
        with col1:
            area = st.selectbox("Area", options=AREAS, index=0)
            item = st.selectbox("Item (Crop Type)", options=ITEMS, index=0)
            year = st.number_input("Year", min_value=MIN_YEAR, max_value=MAX_YEAR, value=2022, step=1)
        with col2:
            rainfall = st.number_input("Average Annual Rainfall (mm)", min_value=MIN_RAINFALL, max_value=MAX_RAINFALL, value=1200.0, step=10.0, format="%.1f")
            pesticides = st.number_input("Pesticides (tonnes)", min_value=MIN_PESTICIDES, max_value=MAX_PESTICIDES, value=60000.0, step=100.0, format="%.1f")
            temperature = st.number_input("Average Temperature (°C)", min_value=MIN_TEMP, max_value=MAX_TEMP, value=25.5, step=0.1, format="%.1f")

        submitted = st.form_submit_button("Get Prediction")

        if submitted:
            input_data = {
                "Area": area,
                "Item": item,
                "Year": year,
                "rainfall": rainfall,
                "pesticides": pesticides,
                "temperature": temperature
            }
            input_df = pd.DataFrame([input_data])

            if crop_yield_pipeline:
                predicted_yields = predict_yield_helper(input_df, crop_yield_pipeline)
                if predicted_yields:
                    predicted_yield = predicted_yields[0]
                    st.success(f"Predicted Crop Yield: **{predicted_yield:,.0f} hg/ha**")
                    st.balloons()
                else:
                    st.error("Prediction could not be made.")
            else:
                st.warning("Crop Yield Model not available. Please check the logs.")

with tab2:
    st.subheader("Pest Detection")
    st.write("Upload an image of a crop or pest to get a detection from our AI model.")

    uploaded_file = st.file_uploader("Choose an image...", type=["jpg", "jpeg", "png"])

    if uploaded_file is not None:
        st.image(uploaded_file, caption="Uploaded Image", use_column_width=True)
        st.write("")
        st.write("Detecting pest...")

        image_bytes = uploaded_file.getvalue()

        if pest_detection_model:
            detections = detect_pest_helper(image_bytes, pest_detection_model)
            if detections:
                st.success("Detection Results:")
                for detection in detections:
                    st.write(f"- **{detection['description']}** (Confidence: {detection['probability']:.2f})")
            else:
                st.info("No significant detections found.")
        else:
            st.warning("Pest Detection Model not available. Please check the logs.")

with tab3:
    st.subheader("Farmer Advisory Chat")
    st.markdown("--- Nosindiso AI Demo --- ")

    if "messages" not in st.session_state:
        st.session_state.messages = []

    for message in st.session_state.messages:
        with st.chat_message(message["role"]):
            st.markdown(message["content"])

    if prompt := st.chat_input("How can I help you, farmer?"):
        st.session_state.messages.append({"role": "user", "content": prompt})
        with st.chat_message("user"):
            st.markdown(prompt)

        with st.spinner("Getting advice from the AI..."):
            if gemini_llm:
                response_text = st.session_state.get('gemini_response_cache', {})
                if prompt not in response_text:
                    response_text[prompt] = asyncio.run(get_gemini_response(prompt, gemini_llm))
                    st.session_state['gemini_response_cache'] = response_text

                ai_response = response_text.get(prompt, "Error: Could not get a response from AI.")
                st.session_state.messages.append({"role": "assistant", "content": ai_response})
                with st.chat_message("assistant"):
                    st.markdown(ai_response)
            else:
                error_msg = "AI chat model not available. Please check API key configuration."
                st.error(error_msg)
                st.session_state.messages.append({"role": "assistant", "content": error_msg})
                with st.chat_message("assistant"):
                    st.markdown(error_msg)''')


import streamlit as st
import requests # Will be removed later, kept for initial modification clarity
import json
import pandas as pd
import numpy as np
import pickle # Replaced joblib with pickle
import os
import io
from PIL import Image
import asyncio # Added for async operations

# Imports for Gemini API
import google.genai as genai # Corrected import
from google.colab import userdata

# Imports for Pest Detection
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image

# --- Configuration & Global Variables ---
MODEL_PATH = 'model_artifacts/xgboost_pipeline.joblib'

# --- Caching Models for Efficiency ---
@st.cache_resource
def load_crop_yield_model():
    if not os.path.exists(MODEL_PATH):
        st.error(f"Crop Yield Model file not found at {MODEL_PATH}.")
        return None
    try:
        with open(MODEL_PATH, 'rb'

In [12]:
import joblib
import pandas as pd
import numpy as np

def predict_yield(input_data_df: pd.DataFrame) -> pd.Series:
    """
    Predicts crop yield (hg/ha) for new input data using the saved XGBoost model pipeline.

    Args:
        input_data_df (pd.DataFrame): A DataFrame containing new data with columns
                                      'Area', 'Item', 'Year', 'rainfall', 'pesticides',
                                      and 'temperature'.

    Returns:
        pd.Series: Predicted crop yields in hg/ha (original scale).
    """
    model_path = 'model_artifacts/xgboost_pipeline.joblib'
    try:
        # Load the saved pipeline
        pipeline = joblib.load(model_path)
    except FileNotFoundError:
        print(f"Error: Model pipeline not found at {model_path}. Please ensure it is saved.")
        return pd.Series()
    except Exception as e:
        print(f"An error occurred while loading the model: {e}")
        return pd.Series()

    # Make predictions (pipeline handles preprocessing)
    log_predictions = pipeline.predict(input_data_df)

    # Inverse transform to get predictions in original scale
    original_scale_predictions = np.expm1(log_predictions)

    return pd.Series(original_scale_predictions, index=input_data_df.index)

# --- Test the prediction function with sample data ---
print("\n--- Testing the predict_yield function ---")

# Create a sample DataFrame for prediction
sample_data = pd.DataFrame({
    'Area': ['India', 'Brazil', 'France', 'India'],
    'Item': ['Wheat', 'Maize', 'Potatoes', 'Rice, paddy'],
    'Year': [2022, 2023, 2022, 2021],
    'rainfall': [1200.0, 1800.0, 700.0, 1100.0],
    'pesticides': [60000.0, 80000.0, 30000.0, 55000.0],
    'temperature': [25.5, 28.0, 15.0, 26.5]
})

print("Sample Input Data:")
print(sample_data)

# Get predictions
predicted_yields = predict_yield(sample_data)

print("\nPredicted Crop Yields (hg/ha):")
print(predicted_yields.apply(lambda x: f'{x:,.0f}'))
print("\n--- Prediction function tested successfully ---")



--- Testing the predict_yield function ---
Sample Input Data:
     Area         Item  Year  rainfall  pesticides  temperature
0   India        Wheat  2022    1200.0     60000.0         25.5
1  Brazil        Maize  2023    1800.0     80000.0         28.0
2  France     Potatoes  2022     700.0     30000.0         15.0
3   India  Rice, paddy  2021    1100.0     55000.0         26.5

Predicted Crop Yields (hg/ha):
0     59,386
1     16,573
2    156,223
3     20,197
dtype: object

--- Prediction function tested successfully ---


In [13]:
print('Checking for running FastAPI process (uvicorn)...')
!ps aux | grep uvicorn

Checking for running FastAPI process (uvicorn)...
root       21049  0.0  0.0   7372  3432 ?        S    10:30   0:00 /bin/bash -c ps aux | grep uvicorn
root       21051  0.0  0.0   6480  2564 ?        S    10:30   0:00 grep uvicorn


In [14]:
# First, install Streamlit if you haven't already
!pip install streamlit

import streamlit as st
import requests
import json
import pandas as pd
import numpy as np

# --- Configuration --- #
# Adjust this URL if your FastAPI is running on a different host or port,
# or if you are using a tool like ngrok to expose it.
# For local Colab environment, it's usually http://127.0.0.1:8000
API_BASE_URL = "http://127.0.0.1:8000"
PREDICT_API_URL = f"{API_BASE_URL}/predict"
CHAT_API_URL = f"{API_BASE_URL}/chat"

# --- Streamlit App --- #
st.set_page_config(page_title="Crop Yield Predictor & Advisory", layout="centered")

st.title("Crop Yield Prediction & Farmer Advisory")
st.markdown("Use the sections below to predict crop yields and get AI-powered advice.")

st.subheader("Input Parameters for Yield Prediction")

# Get unique values for Area and Item from the original DataFrame `df`
# These should ideally be consistent with the training data.
# In a real deployment, these lists would be pre-loaded or fetched from a config.

# Hardcoding based on the user's updated data for 'Area'
AREAS = ['Zambia', 'Zimbabwe']
ITEMS = ['Wheat', 'Maize', 'Potatoes', 'Rice, paddy', 'Sorghum', 'Soybeans', 'Cassava', 'Sweet potatoes']

# MIN/MAX values are derived from the original dataset and are generally stable
MIN_YEAR = 2020
MAX_YEAR = 2030
MIN_RAINFALL = 0.0
MAX_RAINFALL = 3500.0
MIN_PESTICIDES = 0.0
MAX_PESTICIDES = 150000.0
MIN_TEMP = -20.0
MAX_TEMP = 40.0


with st.form("prediction_form"):
    col1, col2 = st.columns(2)
    with col1:
        area = st.selectbox("Area", options=AREAS, index=0)
        item = st.selectbox("Item (Crop Type)", options=ITEMS, index=0)
        year = st.number_input("Year", min_value=MIN_YEAR, max_value=MAX_YEAR, value=2022, step=1)
    with col2:
        rainfall = st.number_input("Average Annual Rainfall (mm)", min_value=MIN_RAINFALL, max_value=MAX_RAINFALL, value=1200.0, step=10.0, format="%.1f")
        pesticides = st.number_input("Pesticides (tonnes)", min_value=MIN_PESTICIDES, max_value=MAX_PESTICIDES, value=60000.0, step=100.0, format="%.1f")
        temperature = st.number_input("Average Temperature (°C)", min_value=MIN_TEMP, max_value=MAX_TEMP, value=25.5, step=0.1, format="%.1f")

    submitted = st.form_submit_button("Get Prediction")

    if submitted:
        input_data = {
            "Area": area,
            "Item": item,
            "Year": year,
            "rainfall": rainfall,
            "pesticides": pesticides,
            "temperature": temperature
        }

        # API expects a list of CropYieldRequest objects
        payload = [input_data]

        try:
            response = requests.post(PREDICT_API_URL, headers={'Content-Type': 'application/json'}, data=json.dumps(payload))
            response.raise_for_status() # Raise an exception for HTTP errors

            predictions = response.json()
            predicted_yield = predictions['predictions'][0] # Assuming a single prediction

            st.success(f"Predicted Crop Yield: **{predicted_yield:,.0f} hg/ha**")
            st.balloons()

        except requests.exceptions.ConnectionError:
            st.error(f"Error: Could not connect to the API at {PREDICT_API_URL}. Please ensure your FastAPI server is running.")
        except requests.exceptions.RequestException as e:
            st.error(f"An error occurred during the API request: {e}")
            if response is not None:
                st.error(f"API Response Status: {response.status_code}")
                st.error(f"API Response Body: {response.text}")
        except json.JSONDecodeError:
            st.error("Error: Could not decode JSON from API response.")
            if response is not None:
                st.error(f"API Response Body: {response.text}")
        except Exception as e:
            st.error(f"An unexpected error occurred: {e}")

st.markdown("--- Nosindiso AI Demo --- ")

st.subheader("Farmer Advisory Chat")

# Initialize chat history
if "messages" not in st.session_state:
    st.session_state.messages = []

# Display chat messages from history on app rerun
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# React to user input
if prompt := st.chat_input("How can I help you, farmer?"):
    # Display user message in chat message container
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    # Display assistant response in chat message container
    with st.spinner("Getting advice from the AI..."):
        try:
            chat_payload = {"message": prompt}
            chat_response = requests.post(
                CHAT_API_URL,
                headers={"Content-Type": "application/json"},
                data=json.dumps(chat_payload)
            )
            chat_response.raise_for_status() # Raise an exception for HTTP errors

            ai_response = chat_response.json().get("response", "Error: No response from AI.")
            st.session_state.messages.append({"role": "assistant", "content": ai_response})
            with st.chat_message("assistant"):
                st.markdown(ai_response)
        except requests.exceptions.ConnectionError:
            error_msg = f"Error: Could not connect to the API at {CHAT_API_URL}. Please ensure your FastAPI server is running and accessible."
            st.error(error_msg)
            st.session_state.messages.append({"role": "assistant", "content": error_msg})
            with st.chat_message("assistant"):
                st.markdown(error_msg)
        except requests.exceptions.RequestException as e:
            error_msg = f"An error occurred during the chat API request: {e}"
            st.error(error_msg)
            if chat_response is not None:
                st.error(f"API Response Status: {chat_response.status_code}")
                st.error(f"API Response Body: {chat_response.text}")
            st.session_state.messages.append({"role": "assistant", "content": error_msg})
            with st.chat_message("assistant"):
                st.markdown(error_msg)
        except json.JSONDecodeError:
            error_msg = "Error: Could not decode JSON from chat API response."
            st.error(error_msg)
            if chat_response is not None:
                st.error(f"API Response Body: {chat_response.text}")
            st.session_state.messages.append({"role": "assistant", "content": error_msg})
            with st.chat_message("assistant"):
                st.markdown(error_msg)
        except Exception as e:
            error_msg = f"An unexpected error occurred during chat: {e}"
            st.error(error_msg)
            st.session_state.messages.append({"role": "assistant", "content": error_msg})
            with st.chat_message("assistant"):
                st.markdown(error_msg)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 97.0 MB/s eta 0:00:00


2026-04-05 10:30:17.898 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 10:30:17.900 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 10:30:18.836 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-04-05 10:30:18.841 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 10:30:18.844 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 10:30:18.849 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 10:30:18.857 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

In [15]:
# Install the Google Generative AI library
!pip install -q -U google-generativeai

# Import the Python SDK
import google.generativeai as genai
# Used to securely store your API key
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('yield_key')
genai.configure(api_key=GOOGLE_API_KEY)

In [16]:
# Initialize the Gemini API
# We'll use a suitable model for chat, for example, 'gemini-pro'
gemini_model = genai.GenerativeModel('gemini-pro')

In [17]:
pip install google genai

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.8/831.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 3.0 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for tiktoken (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tiktoken
Failed to build tiktoken
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tiktoken)


In [18]:
print("Displaying Streamlit logs from streamlit_log.txt:")
try:
    with open('streamlit_log.txt', 'r') as f:
        print(f.read())
except FileNotFoundError:
    print("Error: 'streamlit_log.txt' not found. Streamlit may not have been launched yet or logs were not written.")
except Exception as e:
    print(f"An error occurred while reading the log file: {e}")

Displaying Streamlit logs from streamlit_log.txt:
Error: 'streamlit_log.txt' not found. Streamlit may not have been launched yet or logs were not written.


In [19]:
# Reinstall google-generativeai and tiktoken to ensure they build correctly with the new tools
# Using --upgrade to ensure a fresh install attempt
print("Reinstalling google-generativeai and tiktoken...")
!pip install --upgrade google-generativeai tiktoken
print("Reinstallation of google-generativeai and tiktoken attempted.")

Reinstalling google-generativeai and tiktoken...
Reinstallation of google-generativeai and tiktoken attempted.


In [20]:
!pip install tiktoken --no-build-isolation --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.1/801.1 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.7/153.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 7.2 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: regex
    Found existing installation: regex 2025.11.3
    Uninstalling regex-2025.11.3:
      Successful

In [21]:
pip install google-genai

In [22]:
%%writefile main.py
from pydantic import BaseModel, Field
from typing import List, Dict
import pandas as pd
import numpy as np
import joblib
import os
import io
from PIL import Image
from datetime import datetime, timedelta

# FastAPI imports
from fastapi import FastAPI, HTTPException, Depends, status, UploadFile, File
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm

# Authentication imports
from passlib.context import CryptContext
from jose import JWTError, jwt

# Import for Gemini API
import google.genai as genai
from google.colab import userdata

# New Tensorflow imports for pest detection (COMMENTED OUT)
# import tensorflow as tf
# from tensorflow.keras.applications import MobileNetV2
# from tensorflow.keras.applications.mobilenet_v2 import preprocess_input, decode_predictions
# from tensorflow.keras.preprocessing import image

# Database imports
from sqlalchemy import create_engine, Column, Integer, String, Boolean
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, Session

# --- Configure TensorFlow to use CPU only to avoid CUDA errors (COMMENTED OUT) ---
# tf.config.set_visible_devices([], 'GPU')

# --- 0. Configure Gemini & Security ---
# Retrieve GOOGLE_API_KEY from environment variable
# YIELD_API_KEY will be set via os.environ in the launcher script.
GOOGLE_API_KEY = os.getenv('YIELD_API_KEY')
if GOOGLE_API_KEY:
    genai.configure(api_key=GOOGLE_API_KEY)

# Security settings
SECRET_KEY = os.getenv("SECRET_KEY", "a_very_secret_key_that_should_be_changed_in_production")
ALGORITHM = "HS256"
ACCESS_TOKEN_EXPIRE_MINUTES = 30

pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="token") # This tokenUrl points to our /login endpoint

# --- Database Setup ---
SQLALCHEMY_DATABASE_URL = "sqlite:///./sql_app.db"
engine = create_engine(SQLALCHEMY_DATABASE_URL, connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base = declarative_base()

# Dependency to get the DB session
def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

# --- SQLAlchemy Models ---
class DBUser(Base):
    __tablename__ = "users"

    id = Column(Integer, primary_key=True, index=True)
    username = Column(String, unique=True, index=True)
    email = Column(String, unique=True, index=True, nullable=True)
    full_name = Column(String, nullable=True)
    hashed_password = Column(String)
    disabled = Column(Boolean, default=False)

# --- 1. Pydantic Models ---

# User models
class User(BaseModel):
    username: str
    email: str | None = None
    full_name: str | None = None
    disabled: bool | None = None

class UserInDB(User):
    hashed_password: str

class Token(BaseModel):
    access_token: str
    token_type: str

class TokenData(BaseModel):
    username: str | None = None

class CropYieldRequest(BaseModel):
    Area: str
    Item: str
    Year: int = Field(..., gt=0)
    rainfall: float = Field(..., ge=0.0)
    pesticides: float = Field(..., ge=0.0)
    temperature: float = Field(..., ge=-50.0, le=70.0)

class ChatRequest(BaseModel):
    message: str

# --- 2. FastAPI App Instance ---
app = FastAPI(title="Crop Yield & Farmer Advisory API")

# --- 3. User Management Functions ---

def get_password_hash(password):
    return pwd_context.hash(password)

def verify_password(plain_password, hashed_password):
    return pwd_context.verify(plain_password, hashed_password)

def create_access_token(data: dict, expires_delta: timedelta | None = None):
    to_encode = data.copy()
    if expires_delta:
        expire = datetime.utcnow() + expires_delta
    else:
        expire = datetime.utcnow() + timedelta(minutes=ACCESS_TOKEN_EXPIRE_MINUTES)
    to_encode.update({"exp": expire})
    encoded_jwt = jwt.encode(to_encode, SECRET_KEY, algorithm=ALGORITHM)
    return encoded_jwt

async def get_current_user(token: str = Depends(oauth2_scheme), db: Session = Depends(get_db)):
    credentials_exception = HTTPException(
        status_code=status.HTTP_401_UNAUTHORIZED,
        detail="Could not validate credentials",
        headers={"WWW-Authenticate": "Bearer"},
    )
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        username: str = payload.get("sub")
        if username is None:
            raise credentials_exception
        token_data = TokenData(username=username)
    except JWTError:
        raise credentials_exception
    user = db.query(DBUser).filter(DBUser.username == token_data.username).first()
    if user is None:
        raise credentials_exception
    return UserInDB(**user.__dict__)

# --- 4. Global Models ---
model_pipeline = None
gemini_model = None
pest_model = None # Set to None as MobileNetV2 is commented out
MODEL_PATH = 'model_artifacts/xgboost_pipeline.joblib'

@app.on_event('startup')
async def load_models():
    global model_pipeline, gemini_model, pest_model

    # Create database tables
    Base.metadata.create_all(bind=engine)

    # Add a default test user if not present in the DB
    db = SessionLocal()
    TEST_USERNAME = "testuser"
    TEST_PASSWORD = "testpassword123"
    db_user = db.query(DBUser).filter(DBUser.username == TEST_USERNAME).first()
    if not db_user:
        hashed_password = get_password_hash(TEST_PASSWORD)
        new_user = DBUser(username=TEST_USERNAME, hashed_password=hashed_password)
        db.add(new_user)
        db.commit()
        db.refresh(new_user)
        print(f"Default test user '{TEST_USERNAME}' registered during startup.")
    db.close()

    if not os.path.exists(MODEL_PATH):
        print(f"Warning: Model file not found at {MODEL_PATH}. Prediction endpoint might not work.")
        model_pipeline = None
    else:
        try:
            model_pipeline = joblib.load(MODEL_PATH)
        except Exception as e:
            print(f"Error loading crop yield model: {e}")
            model_pipeline = None

    if GOOGLE_API_KEY:
        try:
            gemini_model = genai.GenerativeModel('gemini-pro')
        except Exception as e:
            print(f"Error loading Gemini model: {e}")
            gemini_model = None
    else:
        print("Warning: YIELD_API_KEY not set. Chat endpoint might not work.")
        gemini_model = None

    # Try to load pest_model (COMMENTED OUT)
    # try:
    #     pest_model = MobileNetV2(weights='imagenet')
    # except Exception as e:
    #     print(f"Error loading MobileNetV2 model: {e}")
    #     pest_model = None

    print("All models attempted to load.")

# --- 5. API Endpoints ---

@app.get('/')
async def read_root():
    return {'status': 'active', 'message': 'API is running'}

@app.post('/predict')
async def predict(request_data: List[CropYieldRequest], current_user: User = Depends(get_current_user)):
    if model_pipeline is None: raise HTTPException(status_code=500, detail="Model not loaded")
    df = pd.DataFrame([item.model_dump() for item in request_data])
    log_predictions = model_pipeline.predict(df)
    return {'predictions': np.expm1(log_predictions).tolist()}

@app.post('/chat')
async def chat(request: ChatRequest, current_user: User = Depends(get_current_user)):
    if not gemini_model:
        raise HTTPException(status_code=500, detail="Gemini API not configured")
    response = await gemini_model.generate_content_async(request.message)
    return {'response': response.text}

@app.post('/detect_pest')
async def detect_pest_endpoint(file: UploadFile = File(...), current_user: User = Depends(get_current_user)):
    # if pest_model is None: raise HTTPException(status_code=500, detail="Pest detection model not loaded")
    # image_bytes = await file.read()
    # img = Image.open(io.BytesIO(image_bytes)).resize((224, 224))
    # x = preprocess_input(np.expand_dims(image.img_to_array(img), axis=0))
    # preds = decode_predictions(pest_model.predict(x), top=3)[0]
    # return {"detections": [{"label": l, "description": d, "probability": float(p)} for (_, l, p) in preds]}
    raise HTTPException(status_code=501, detail="Pest detection temporarily disabled") # Return 501 Not Implemented

@app.post("/token", response_model=Token)
async def login_for_access_token(form_data: OAuth2PasswordRequestForm = Depends(), db: Session = Depends(get_db)):
    user_data = db.query(DBUser).filter(DBUser.username == form_data.username).first()
    if not user_data or not verify_password(form_data.password, user_data.hashed_password):
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Incorrect username or password",
            headers={"WWW-Authenticate": "Bearer"},
        )
    user = UserInDB(**user_data.__dict__)
    access_token_expires = timedelta(minutes=ACCESS_TOKEN_EXPIRE_MINUTES)
    access_token = create_access_token(
        data={"sub": user.username, "scopes": form_data.scopes},
        expires_delta=access_token_expires,
    )
    return {"access_token": access_token, "token_type": "bearer"}

@app.post("/register_user")
async def register_user(user: UserInDB, db: Session = Depends(get_db)):
    db_user = db.query(DBUser).filter(DBUser.username == user.username).first()
    if db_user:
        raise HTTPException(status_code=400, detail="Username already registered")

    hashed_password = get_password_hash(user.hashed_password)
    new_db_user = DBUser(username=user.username, email=user.email, full_name=user.full_name, hashed_password=hashed_password)
    db.add(new_db_user)
    db.commit()
    db.refresh(new_db_user)
    return {"message": "User registered successfully", "username": new_db_user.username}

Overwriting main.py


In [10]:
!pip install sqlalchemy

In [9]:
import time
time.sleep(10) # Give Streamlit some time to start up and write logs

print("Displaying fresh Streamlit logs from streamlit_log.txt:")
try:
    with open('streamlit_log.txt', 'r') as f:
        print(f.read())
except FileNotFoundError:
    print("Error: 'streamlit_log.txt' not found after relaunch. Streamlit might not have started or logs were not written.")
except Exception as e:
    print(f"An error occurred while reading the log file: {e}")

Displaying fresh Streamlit logs from streamlit_log.txt:
2026-04-05 11:08:53.373 
'server.enableXsrfProtection=true'.
As a result, 'server.enableCORS' is being overridden to 'true'.

More information:
In order to protect against CSRF attacks, we send a cookie with each request.
To do so, we must specify allowable origins, which places a restriction on
cross-origin resource sharing.

If cross origin resource sharing is required, please disable server.enableXsrfProtection.
            

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.28.16.248:8501




In [ ]:
import sys
python_path = sys.executable
print(f"Python executable path: {python_path}")

In [1]:
%%writefile streamlit_app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import os
import io
from PIL import Image
import asyncio
from datetime import datetime, timedelta

# Authentication imports
from passlib.context import CryptContext

# Database imports
from sqlalchemy import create_engine, Column, Integer, String, Boolean
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, Session

# Import for Gemini API
import google.genai as genai
from google.colab import userdata

# Imports for Pest Detection
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image

# --- Configuration & Global Variables ---
MODEL_PATH = 'model_artifacts/xgboost_pipeline.joblib'

# Security settings (local to Streamlit now)
pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")

# Database Setup (local to Streamlit now)
SQLALCHEMY_DATABASE_URL = "sqlite:///./sql_app.db"
engine = create_engine(SQLALCHEMY_DATABASE_URL, connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base = declarative_base()

# --- SQLAlchemy Models ---
class DBUser(Base):
    __tablename__ = "users"

    id = Column(Integer, primary_key=True, index=True)
    username = Column(String, unique=True, index=True)
    email = Column(String, unique=True, index=True, nullable=True)
    full_name = Column(String, nullable=True)
    hashed_password = Column(String)
    disabled = Column(Boolean, default=False)

# --- User Management Functions ---
def get_password_hash(password):
    return pwd_context.hash(password)

def verify_password(plain_password, hashed_password):
    return pwd_context.verify(plain_password, hashed_password)

def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

# --- Caching Models for Efficiency ---
@st.cache_resource
def load_crop_yield_model():
    if not os.path.exists(MODEL_PATH):
        st.error(f"Crop Yield Model file not found at {MODEL_PATH}.")
        return None
    try:
        # Ensure consistent use of joblib or pickle. Since the notebook created it with joblib,
        # we'll stick to joblib here.
        pipeline = joblib.load(MODEL_PATH)
        return pipeline
    except Exception as e:
        st.error(f"Error loading Crop Yield model: {e}")
        return None

@st.cache_resource
def load_pest_detection_model():
    try:
        # Explicitly configure TensorFlow to use CPU only
        tf.config.set_visible_devices([], 'GPU')
        model = MobileNetV2(weights='imagenet')
        return model
    except Exception as e:
        st.error(f"Error loading Pest Detection model: {e}")
        return None

@st.cache_resource
def load_gemini_model():
    try:
        api_key = userdata.get('yield_key')
        if not api_key:
            st.error("Google API Key not found. Please set 'yield_key' in Colab secrets.")
            return None
        model = genai.GenerativeModel('gemini-pro', api_key=api_key)
        return model
    except Exception as e:
        st.error(f"Error initializing Gemini model: {e}")
        return None

# --- Helper Functions ---
def predict_yield_helper(input_data_df: pd.DataFrame, model_pipeline_local) -> list[float]:
    if model_pipeline_local is None:
        st.error("Crop Yield Model is not loaded.")
        return []
    expected_columns = ['Area', 'Item', 'Year', 'rainfall', 'pesticides', 'temperature']
    df_processed = input_data_df[expected_columns]
    log_predictions = model_pipeline_local.predict(df_processed)
    original_scale_predictions = np.expm1(log_predictions)
    return original_scale_predictions.tolist()

def detect_pest_helper(image_bytes: bytes, pest_model_local) -> list[dict]:
    if pest_model_local is None:
        st.error("Pest Detection Model is not loaded.")
        return []
    try:
        img = Image.open(io.BytesIO(image_bytes)).resize((224, 224))
        x = image.img_to_array(img)
        x = np.expand_dims(x, axis=0)
        x = preprocess_input(x)
        preds = pest_model_local.predict(x)
        decoded_preds = decode_predictions(preds, top=3)[0]
        results = [{"label": label, "description": description, "probability": float(prob)}
                   for (_, label, prob) in decoded_preds]
        return results
    except Exception as e:
        st.error(f"Image processing or pest detection failed: {e}")
        return []

async def get_gemini_response(message: str, gemini_model_local) -> str:
    if gemini_model_local is None:
        st.error("Gemini model is not initialized.")
        return "Error: Gemini model not available."
    try:
        response = await gemini_model_local.generate_content_async(message)
        return response.text
    except Exception as e:
        st.error(f"Gemini API call failed: {e}")
        return "Error: Could not get a response from AI."

# --- Main App Logic ---
@st.cache_resource
def initialize_database():
    Base.metadata.create_all(bind=engine)
    db = SessionLocal()
    TEST_USERNAME = "testuser"
    TEST_PASSWORD = "testpassword123"
    db_user = db.query(DBUser).filter(DBUser.username == TEST_USERNAME).first()
    if not db_user:
        hashed_password = get_password_hash(TEST_PASSWORD)
        new_user = DBUser(username=TEST_USERNAME, hashed_password=hashed_password)
        db.add(new_user)
        db.commit()
        db.refresh(new_user)
        st.success(f"Default test user '{TEST_USERNAME}' registered during startup.")
    db.close()

initialize_database()

crop_yield_pipeline = load_crop_yield_model()
pest_detection_model = load_pest_detection_model()
gemini_llm = load_gemini_model()

st.set_page_config(page_title='Farmer Advisor & Marketplace', layout='wide')
st.title('🌾 Crop Advisor & Marketplace')

# --- Authentication Logic (Direct DB Interaction) ---
if 'logged_in' not in st.session_state:
    st.session_state.logged_in = False
    st.session_state.username = None

def login_form():
    st.subheader("Login")
    with st.form("login_form"):
        username = st.text_input("Username")
        password = st.text_input("Password", type="password")
        submitted = st.form_submit_button("Login")

        if submitted:
            db = next(get_db())
            user_data = db.query(DBUser).filter(DBUser.username == username).first()
            db.close()
            if user_data and verify_password(password, user_data.hashed_password):
                st.session_state.logged_in = True
                st.session_state.username = username
                st.success(f"Welcome, {username}!")
                st.experimental_rerun()
            else:
                st.error("Incorrect username or password.")

def register_form():
    st.subheader("Register New User")
    with st.form("register_form"):
        new_username = st.text_input("New Username")
        new_password = st.text_input("New Password", type="password")
        confirm_password = st.text_input("Confirm Password", type="password")
        submitted = st.form_submit_button("Register")

        if submitted:
            if new_password != confirm_password:
                st.error("Passwords do not match.")
            elif len(new_password) < 6:
                st.error("Password must be at least 6 characters long.")
            else:
                db = next(get_db())
                db_user = db.query(DBUser).filter(DBUser.username == new_username).first()
                if db_user:
                    st.error("Username already registered")
                    db.close()
                else:
                    hashed_password = get_password_hash(new_password)
                    new_db_user = DBUser(username=new_username, hashed_password=hashed_password)
                    db.add(new_db_user)
                    db.commit()
                    db.refresh(new_db_user)
                    db.close()
                    st.success("Registration successful! Please login.")
                    st.session_state.logged_in = False
                    st.experimental_rerun()

if not st.session_state.logged_in:
    st.sidebar.title("Authentication")
    auth_option = st.sidebar.radio("", ["Login", "Register"])
    if auth_option == "Login":
        login_form()
    else:
        register_form()
    st.stop() # Stop execution if not logged in

# --- Main App Content (only shown if logged in) ---
st.sidebar.write(f"Logged in as: {st.session_state.username}") # Role not strictly needed here
if st.sidebar.button("Logout"):
    st.session_state.logged_in = False
    st.session_state.username = None
    st.experimental_rerun()

tabs = st.tabs(['📈 Yield Prediction', '🐛 Pest Detection', '🦾 AI Advisor', '🛒 Market & Trends'])

with tabs[0]:
    st.header('Yield Prediction')
    with st.form('yield_form'):
        area = st.selectbox('Area', ['Zambia', 'Zimbabwe'])
        item = st.selectbox('Crop', ['Wheat', 'Maize', 'Potatoes', 'Rice, paddy', 'Sorghum', 'Soybeans'])
        year = st.number_input('Year', 2024, 2030, 2025)
        rain = st.slider('Rainfall (mm)', 0, 3500, 1000)
        pest = st.slider('Pesticides (tonnes)', 0, 150000, 5000)
        temp = st.slider('Temp (°C)', 10, 45, 25)
        if st.form_submit_button('Predict'):
            input_data = {
                "Area": area,
                "Item": item,
                "Year": year,
                "rainfall": float(rain),
                "pesticides": float(pest),
                "temperature": float(temp)
            }
            input_df = pd.DataFrame([input_data])
            if crop_yield_pipeline:
                predicted_yields = predict_yield_helper(input_df, crop_yield_pipeline)
                if predicted_yields:
                    predicted_yield = predicted_yields[0]
                    st.success(f'Estimated Yield: {predicted_yield:,.2f} hg/ha')
                else:
                    st.error("Prediction could not be made.")
            else:
                st.warning("Crop Yield Model not available. Please check the logs.")

with tabs[1]:
    st.header('Pest Identification')
    file = st.file_uploader('Upload leaf or pest image', type=['jpg', 'png'])
    if file:
        st.image(file, caption="Uploaded Image", use_column_width=True)
        st.write('Detecting pest...')
        image_bytes = file.getvalue()
        if pest_detection_model:
            detections = detect_pest_helper(image_bytes, pest_detection_model)
            if detections:
                st.success("Detection Results:")
                for detection in detections:
                    st.write(f"- **{detection['description']}** (Confidence: {detection['probability']:.2f})")
            else:
                st.info("No significant detections found.")
        else:
            st.warning("Pest Detection Model not available. Please check the logs.")

with tabs[2]:
    st.header('AI Chatbot')
    if 'messages' not in st.session_state: st.session_state.messages = []
    for m in st.session_state.messages: st.chat_message(m['role']).write(m['content'])
    if prompt := st.chat_input('How can I improve my soil?'):
        st.session_state.messages.append({'role': 'user', 'content': prompt})
        st.chat_message('user').write(prompt)
        with st.spinner("Getting advice from the AI..."):
            if gemini_llm:
                response_text = asyncio.run(get_gemini_response(prompt, gemini_llm))
                st.session_state.messages.append({'role': 'assistant', 'content': response_text})
                st.chat_message('assistant').write(response_text)
            else:
                error_msg = "AI chat model not available. Please check API key configuration."
                st.error(error_msg)
                st.session_state.messages.append({'role': 'assistant', 'content': error_msg})
                st.chat_message('assistant').write(error_msg)

with tabs[3]:
    st.header('🛒 Farmer Marketplace & Trending Crops')
    col1, col2 = st.columns(2)
    with col1:
        st.subheader('Trending Crops This Season')
        MARKET_PATH = 'market_trends.csv'
        if os.path.exists(MARKET_PATH):
            df_trends = pd.read_csv(MARKET_PATH)
            st.dataframe(df_trends, use_container_width=True)
        else: st.info('No trend data available.')
    with col2:
        st.subheader('List Your Crop for Sale')
        with st.form('market_form'):
            seller_name = st.text_input('Name')
            crop_type = st.selectbox('Crop', ['Wheat', 'Maize', 'Potatoes', 'Rice, paddy', 'Sorghum', 'Soybeans'])
            quantity = st.number_input('Quantity (kg)', min_value=1)
            price = st.number_input('Asking Price ($)', min_value=1)
            if st.form_submit_button('Post Listing'):
                st.success(f'Listing created for {seller_name}! Others can now see your {crop_type}.')


Overwriting streamlit_app.py


In [ ]:
import joblib
import os

print(f"'joblib' library is installed and located at: {os.path.dirname(joblib.__file__)}")
print(f"Version of joblib: {joblib.__version__}")

# You can also use help(joblib) to see its documentation.
# help(joblib)


In [18]:
# Reinstall tensorflow to ensure compatible dependencies, especially protobuf
print("Reinstalling tensorflow to resolve potential protobuf conflicts...")
!pip install --upgrade --force-reinstall tensorflow

import streamlit as st
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input, decode_predictions

@st.cache_resource
def load_pest_detection_model():
    try:
        # Ensure weights are downloaded or available
        model = MobileNetV2(weights='imagenet')
        return model
    except Exception as e:
        st.error(f"Error loading Pest Detection model: {e}")
        return None

pest_detection_model = load_pest_detection_model()

Reinstalling tensorflow to resolve potential protobuf conflicts...
  Using cached packaging-26.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
  Using cached wheel-0.46.3-py3-none-any.whl.metadata (2.4 kB)
  Using cached charset_normalizer-3.4.7-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 788.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

KeyboardInterrupt: 

In [17]:
!pip install -q -U google-generativeai google-cloud-aiplatform

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 69.4 MB/s eta 0:00:00


In [16]:
import pandas as pd

# Create a sample market and trends dataset
market_data = {
    'Crop': ['Wheat', 'Maize', 'Potatoes', 'Soybeans'],
    'Current_Price_hg_ha': [450, 320, 150, 600],
    'Trend': ['Upward', 'Stable', 'Downward', 'Upward'],
    'Season': ['Winter', 'Summer', 'All', 'Summer']
}

pd.DataFrame(market_data).to_csv('market_trends.csv', index=False)
print("Market trends data created.")

Market trends data created.


In [2]:
%%writefile streamlit_app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import os
import io
from PIL import Image
import asyncio
from datetime import datetime, timedelta

# Authentication imports
from passlib.context import CryptContext

# Database imports
from sqlalchemy import create_engine, Column, Integer, String, Boolean
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, Session

# Import for Gemini API
import google.generativeai as genai

# Imports for Pest Detection
# Moved imports to inside the load_pest_detection_model function to ensure 'tf' is defined when used.

# --- Configuration & Global Variables ---
MODEL_PATH = 'model_artifacts/xgboost_pipeline.joblib'
MARKET_PATH = 'market_trends.csv'

# Security settings (local to Streamlit now)
pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")

# Database Setup (local to Streamlit now)
SQLALCHEMY_DATABASE_URL = "sqlite:///./sql_app.db"
engine = create_engine(SQLALCHEMY_DATABASE_URL, connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base = declarative_base()

# --- SQLAlchemy Models ---
class DBUser(Base):
    __tablename__ = "users"

    id = Column(Integer, primary_key=True, index=True)
    username = Column(String, unique=True, index=True)
    email = Column(String, unique=True, index=True, nullable=True)
    full_name = Column(String, nullable=True)
    hashed_password = Column(String)
    disabled = Column(Boolean, default=False)

# --- User Management Functions ---
def get_password_hash(password):
    return pwd_context.hash(password)

def verify_password(plain_password, hashed_password):
    return pwd_context.verify(plain_password, hashed_password)

def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

# --- Caching Models for Efficiency ---
@st.cache_resource
def load_crop_yield_model():
    if not os.path.exists(MODEL_PATH):
        st.error(f"Crop Yield Model file not found at {MODEL_PATH}.")
        return None
    try:
        pipeline = joblib.load(MODEL_PATH)
        return pipeline
    except Exception as e:
        st.error(f"Error loading Crop Yield model: {e}")
        return None

@st.cache_resource
def load_pest_detection_model():
    try:
        # Imports moved here to ensure 'tf' is defined within this function's scope.
        import tensorflow as tf
        from tensorflow.keras.applications import MobileNetV2
        from tensorflow.keras.applications.mobilenet_v2 import preprocess_input, decode_predictions
        from tensorflow.keras.preprocessing import image

        # Explicitly configure TensorFlow to use CPU only
        tf.config.set_visible_devices([], 'GPU')
        model = MobileNetV2(weights='imagenet')
        return model
    except Exception as e:
        st.error(f"Error loading Pest Detection model: {e}")
        return None

@st.cache_resource
def load_gemini_model():
    try:
        api_key = os.getenv('YIELD_API_KEY') # Retrieve from environment variable
        if not api_key:
            st.error("Google API Key not found. Please ensure 'YIELD_API_KEY' environment variable is set.")
            return None
        genai.configure(api_key=api_key)
        model = genai.GenerativeModel('gemini-pro')
        return model
    except Exception as e:
        st.error(f"Error initializing Gemini model: {e}")
        return None

# --- Helper Functions ---
def predict_yield_helper(input_data_df: pd.DataFrame, model_pipeline_local) -> list[float]:
    if model_pipeline_local is None:
        st.error("Crop Yield Model is not loaded.")
        return []
    expected_columns = ['Area', 'Item', 'Year', 'rainfall', 'pesticides', 'temperature']
    df_processed = input_data_df[expected_columns]
    log_predictions = model_pipeline_local.predict(df_processed)
    original_scale_predictions = np.expm1(log_predictions)
    return original_scale_predictions.tolist()

def detect_pest_helper(image_bytes: bytes, pest_model_local) -> list[dict]:
    if pest_model_local is None:
        st.error("Pest Detection Model is not loaded.")
        return []
    try:
        # Ensure tensorflow imports are local to this function for consistency with load_pest_detection_model
        import tensorflow as tf
        from tensorflow.keras.applications.mobilenet_v2 import preprocess_input, decode_predictions
        from tensorflow.keras.preprocessing import image

        img = Image.open(io.BytesIO(image_bytes)).resize((224, 224))
        x = image.img_to_array(img)
        x = np.expand_dims(x, axis=0)
        x = preprocess_input(x)
        preds = pest_model_local.predict(x)
        decoded_preds = decode_predictions(preds, top=3)[0]
        results = [{"label": label, "description": description, "probability": float(prob)}
                   for (_, label, prob) in decoded_preds]
        return results
    except Exception as e:
        st.error(f"Image processing or pest detection failed: {e}")
        return []

async def get_gemini_response(message: str, gemini_model_local) -> str:
    if gemini_model_local is None:
        st.error("Gemini model is not initialized.")
        return "Error: Gemini model not available."
    try:
        response = await gemini_model_local.generate_content_async(message)
        return response.text
    except Exception as e:
        st.error(f"Gemini API call failed: {e}")
        return "Error: Could not get a response from AI."

# --- Main App Logic ---
@st.cache_resource
def initialize_database():
    Base.metadata.create_all(bind=engine)
    db = SessionLocal()
    TEST_USERNAME = "testuser"
    TEST_PASSWORD = "testpassword123"
    db_user = db.query(DBUser).filter(DBUser.username == TEST_USERNAME).first()
    if not db_user:
        hashed_password = get_password_hash(TEST_PASSWORD)
        new_user = DBUser(username=TEST_USERNAME, hashed_password=hashed_password)
        db.add(new_user)
        db.commit()
        db.refresh(new_user)
        st.success(f"Default test user '{TEST_USERNAME}' registered during startup.")
    db.close()

initialize_database()

crop_yield_pipeline = load_crop_yield_model()
pest_detection_model = load_pest_detection_model()
gemini_llm = load_gemini_model()

st.set_page_config(page_title='Farmer Advisor & Marketplace', layout='wide')
st.title('🌾 Crop Advisor & Marketplace')

# --- Authentication Logic (Direct DB Interaction) ---
if 'logged_in' not in st.session_state:
    st.session_state.logged_in = False
    st.session_state.username = None

def login_form():
    st.subheader("Login")
    with st.form("login_form"):
        username = st.text_input("Username")
        password = st.text_input("Password", type="password")
        submitted = st.form_submit_button("Login")

        if submitted:
            db = next(get_db())
            user_data = db.query(DBUser).filter(DBUser.username == username).first()
            db.close()
            if user_data and verify_password(password, user_data.hashed_password):
                st.session_state.logged_in = True
                st.session_state.username = username
                st.success(f"Welcome, {username}!")
                st.experimental_rerun()
            else:
                st.error("Incorrect username or password.")

def register_form():
    st.subheader("Register New User")
    with st.form("register_form"):
        new_username = st.text_input("New Username")
        new_password = st.text_input("New Password", type="password")
        confirm_password = st.text_input("Confirm Password", type="password")
        submitted = st.form_submit_button("Register")

        if submitted:
            if new_password != confirm_password:
                st.error("Passwords do not match.")
            elif len(new_password) < 6:
                st.error("Password must be at least 6 characters long.")
            else:
                db = next(get_db())
                db_user = db.query(DBUser).filter(DBUser.username == new_username).first()
                if db_user:
                    st.error("Username already registered")
                    db.close()
                else:
                    hashed_password = get_password_hash(new_password)
                    new_db_user = DBUser(username=new_username, hashed_password=hashed_password)
                    db.add(new_db_user)
                    db.commit()
                    db.refresh(new_db_user)
                    db.close()
                    st.success("Registration successful! Please login.")
                    st.session_state.logged_in = False
                    st.experimental_rerun()

if not st.session_state.logged_in:
    st.sidebar.title("Authentication")
    auth_option = st.sidebar.radio("", ["Login", "Register"])
    if auth_option == "Login":
        login_form()
    else:
        register_form()
    st.stop() # Stop execution if not logged in

# --- Main App Content (only shown if logged in) ---
st.sidebar.write(f"Logged in as: {st.session_state.username}") # Role not strictly needed here
if st.sidebar.button("Logout"):
    st.session_state.logged_in = False
    st.session_state.username = None
    st.experimental_rerun()

tabs = st.tabs(['📈 Yield Prediction', '🐛 Pest Detection', '🦾 AI Advisor', '🛒 Market & Trends'])

with tabs[0]:
    st.header('Yield Prediction')
    with st.form('yield_form'):
        area = st.selectbox('Area', ['Zambia', 'Zimbabwe'])
        item = st.selectbox('Crop', ['Wheat', 'Maize', 'Potatoes', 'Rice, paddy', 'Sorghum', 'Soybeans'])
        year = st.number_input('Year', 2024, 2030, 2025)
        rain = st.slider('Rainfall (mm)', 0, 3500, 1000)
        pest = st.slider('Pesticides (tonnes)', 0, 150000, 5000)
        temp = st.slider('Temp (°C)', 10, 45, 25)
        if st.form_submit_button('Predict'):
            input_data = {
                "Area": area,
                "Item": item,
                "Year": year,
                "rainfall": float(rain),
                "pesticides": float(pest),
                "temperature": float(temp)
            }
            input_df = pd.DataFrame([input_data])
            if crop_yield_pipeline:
                predicted_yields = predict_yield_helper(input_df, crop_yield_pipeline)
                if predicted_yields:
                    predicted_yield = predicted_yields[0]
                    st.success(f'Estimated Yield: {predicted_yield:,.2f} hg/ha')
                else:
                    st.error("Prediction could not be made.")
            else:
                st.warning("Crop Yield Model not available. Please check the logs.")

with tabs[1]:
    st.header('Pest Identification')
    file = st.file_uploader('Upload leaf or pest image', type=['jpg', 'png'])
    if file:
        st.image(file, caption="Uploaded Image", use_column_width=True)
        st.write('Detecting pest...')
        image_bytes = file.getvalue()
        if pest_detection_model:
            detections = detect_pest_helper(image_bytes, pest_detection_model)
            if detections:
                st.success("Detection Results:")
                for detection in detections:
                    st.write(f"- **{detection['description']}** (Confidence: {detection['probability']:.2f})")
            else:
                st.info("No significant detections found.")
        else:
            st.warning("Pest Detection Model not available. Please check the logs.")

with tabs[2]:
    st.header('AI Chatbot')
    if 'messages' not in st.session_state: st.session_state.messages = []
    for m in st.session_state.messages: st.chat_message(m['role']).write(m['content'])
    if prompt := st.chat_input('How can I improve my soil?'):
        st.session_state.messages.append({'role': 'user', 'content': prompt})
        st.chat_message('user').write(prompt)
        with st.spinner("Getting advice from the AI..."):
            if gemini_llm:
                response_text = asyncio.run(get_gemini_response(prompt, gemini_llm))
                st.session_state.messages.append({'role': 'assistant', 'content': response_text})
                st.chat_message('assistant').write(response_text)
            else:
                error_msg = "AI chat model not available. Please check API key configuration."
                st.error(error_msg)
                st.session_state.messages.append({'role': 'assistant', 'content': error_msg})
                st.chat_message('assistant').write(error_msg)

with tabs[3]:
    st.header('🛒 Farmer Marketplace & Trending Crops')
    col1, col2 = st.columns(2)
    with col1:
        st.subheader('Trending Crops This Season')
        if os.path.exists(MARKET_PATH):
            df_trends = pd.read_csv(MARKET_PATH)
            st.dataframe(df_trends, use_container_width=True)
        else: st.info('No trend data available.')
    with col2:
        st.subheader('List Your Crop for Sale')
        with st.form('market_form'):
            seller_name = st.text_input('Name')
            crop_type = st.selectbox('Crop', ['Wheat', 'Maize', 'Potatoes', 'Rice, paddy', 'Sorghum', 'Soybeans'])
            quantity = st.number_input('Quantity (kg)', min_value=1)
            price = st.number_input('Asking Price ($)', min_value=1)
            if st.form_submit_button('Post Listing'):
                st.success(f'Listing created for {seller_name}! Others can now see your {crop_type}.')

Writing streamlit_app.py


In [75]:
import os

# The api_key variable is already in the kernel's memory from a previous step.
# We will use this existing variable to set the environment variable.
# IMPORTANT: Ensure 'api_key' contains your actual Google API Key.
os.environ['YIELD_API_KEY'] = api_key

print("YIELD_API_KEY environment variable set from existing kernel variable.")

YIELD_API_KEY environment variable set from existing kernel variable.


In [76]:
# Clear existing logs before attempting to restart
print("Clearing previous streamlit_log.txt...")
!echo '' > streamlit_log.txt
print("Previous Streamlit logs cleared.")

Clearing previous streamlit_log.txt...
Previous Streamlit logs cleared.


In [77]:
# Kill any existing Streamlit processes to ensure a clean restart
print("Attempting to kill existing Streamlit processes...")
!pkill -f streamlit || true # '|| true' prevents error if no process is found
print("Existing Streamlit processes killed (if any).")

Attempting to kill existing Streamlit processes...
^C
Existing Streamlit processes killed (if any).


In [78]:
import os
import time

print('Launching Streamlit app in background...')
# Use nohup to ensure the process continues even if the terminal is closed
# Redirect output to streamlit_log.txt for debugging if needed
!nohup streamlit run streamlit_app.py --server.port 8501 --server.enableCORS false --browser.gatherUsageStats false > streamlit_log.txt 2>&1 &

# Add a short delay for Streamlit to start and expose its URL
time.sleep(10)

print("Streamlit app launched. Checking logs for external URL...")

# Display the logs to get the external URL
try:
    with open('streamlit_log.txt', 'r') as f:
        logs = f.read()
        print(logs)
        # Extract and print the External URL from the logs
        external_url = None
        for line in logs.splitlines():
            if "External URL:" in line:
                external_url = line.split("External URL:")[1].strip()
                break
        if external_url:
            print(f"\n\u2705 Your Streamlit app is running! Access it here: {external_url}")
        else:
            print("\u274c Could not find External URL in Streamlit logs. Please check 'streamlit_log.txt' manually.")
except FileNotFoundError:
    print("Error: 'streamlit_log.txt' not found after launching Streamlit. It might not have started or logs were not written.")
except Exception as e:
    print(f"An error occurred while reading the Streamlit log file: {e}")

Launching Streamlit app in background...
Streamlit app launched. Checking logs for external URL...
2026-04-05 11:49:30.064 
'server.enableXsrfProtection=true'.
As a result, 'server.enableCORS' is being overridden to 'true'.

More information:
In order to protect against CSRF attacks, we send a cookie with each request.
To do so, we must specify allowable origins, which places a restriction on
cross-origin resource sharing.

If cross origin resource sharing is required, please disable server.enableXsrfProtection.
            

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.28.16.248:8501



✅ Your Streamlit app is running! Access it here: http://34.28.16.248:8501


In [71]:
import os

# The api_key variable is already in the kernel's memory from a previous step.
# We will use this existing variable to set the environment variable.
# IMPORTANT: Ensure 'api_key' contains your actual Google API Key.
os.environ['YIELD_API_KEY'] = api_key

print("YIELD_API_KEY environment variable set from existing kernel variable.")

YIELD_API_KEY environment variable set from existing kernel variable.


In [72]:
# Clear existing logs before attempting to restart
print("Clearing previous streamlit_log.txt...")
!echo '' > streamlit_log.txt
print("Previous Streamlit logs cleared.")

Clearing previous streamlit_log.txt...
Previous Streamlit logs cleared.


In [73]:
# Kill any existing Streamlit processes to ensure a clean restart
print("Attempting to kill existing Streamlit processes...")
!pkill -f streamlit || true # '|| true' prevents error if no process is found
print("Existing Streamlit processes killed (if any).")

Attempting to kill existing Streamlit processes...
^C
Existing Streamlit processes killed (if any).


In [74]:
import os
import time

print('Launching Streamlit app in background...')
# Use nohup to ensure the process continues even if the terminal is closed
# Redirect output to streamlit_log.txt for debugging if needed
!nohup streamlit run streamlit_app.py --server.port 8501 --server.enableCORS false --browser.gatherUsageStats false > streamlit_log.txt 2>&1 &

# Add a short delay for Streamlit to start and expose its URL
time.sleep(10)

print("Streamlit app launched. Checking logs for external URL...")

# Display the logs to get the external URL
try:
    with open('streamlit_log.txt', 'r') as f:
        logs = f.read()
        print(logs)
        # Extract and print the External URL from the logs
        external_url = None
        for line in logs.splitlines():
            if "External URL:" in line:
                external_url = line.split("External URL:")[1].strip()
                break
        if external_url:
            print(f"\n\u2705 Your Streamlit app is running! Access it here: {external_url}")
        else:
            print("\u274c Could not find External URL in Streamlit logs. Please check 'streamlit_log.txt' manually.")
except FileNotFoundError:
    print("Error: 'streamlit_log.txt' not found after launching Streamlit. It might not have started or logs were not written.")
except Exception as e:
    print(f"An error occurred while reading the Streamlit log file: {e}")

Launching Streamlit app in background...
Streamlit app launched. Checking logs for external URL...
2026-04-05 11:49:15.125 
'server.enableXsrfProtection=true'.
As a result, 'server.enableCORS' is being overridden to 'true'.

More information:
In order to protect against CSRF attacks, we send a cookie with each request.
To do so, we must specify allowable origins, which places a restriction on
cross-origin resource sharing.

If cross origin resource sharing is required, please disable server.enableXsrfProtection.
            

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.28.16.248:8501



✅ Your Streamlit app is running! Access it here: http://34.28.16.248:8501


In [67]:
import os

# The api_key variable is already in the kernel's memory from a previous step.
# We will use this existing variable to set the environment variable.
# IMPORTANT: Ensure 'api_key' contains your actual Google API Key.
os.environ['YIELD_API_KEY'] = api_key

print("YIELD_API_KEY environment variable set from existing kernel variable.")

YIELD_API_KEY environment variable set from existing kernel variable.


In [68]:
# Clear existing logs before attempting to restart
print("Clearing previous streamlit_log.txt...")
!echo '' > streamlit_log.txt
print("Previous Streamlit logs cleared.")

Clearing previous streamlit_log.txt...
Previous Streamlit logs cleared.


In [69]:
# Kill any existing Streamlit processes to ensure a clean restart
print("Attempting to kill existing Streamlit processes...")
!pkill -f streamlit || true # '|| true' prevents error if no process is found
print("Existing Streamlit processes killed (if any).")

Attempting to kill existing Streamlit processes...
^C
Existing Streamlit processes killed (if any).


In [70]:
import os
import time

print('Launching Streamlit app in background...')
# Use nohup to ensure the process continues even if the terminal is closed
# Redirect output to streamlit_log.txt for debugging if needed
!nohup streamlit run streamlit_app.py --server.port 8501 --server.enableCORS false --browser.gatherUsageStats false > streamlit_log.txt 2>&1 &

# Add a short delay for Streamlit to start and expose its URL
time.sleep(10)

print("Streamlit app launched. Checking logs for external URL...")

# Display the logs to get the external URL
try:
    with open('streamlit_log.txt', 'r') as f:
        logs = f.read()
        print(logs)
        # Extract and print the External URL from the logs
        external_url = None
        for line in logs.splitlines():
            if "External URL:" in line:
                external_url = line.split("External URL:")[1].strip()
                break
        if external_url:
            print(f"\n\u2705 Your Streamlit app is running! Access it here: {external_url}")
        else:
            print("\u274c Could not find External URL in Streamlit logs. Please check 'streamlit_log.txt' manually.")
except FileNotFoundError:
    print("Error: 'streamlit_log.txt' not found after launching Streamlit. It might not have started or logs were not written.")
except Exception as e:
    print(f"An error occurred while reading the Streamlit log file: {e}")

Launching Streamlit app in background...
Streamlit app launched. Checking logs for external URL...
2026-04-05 11:48:56.765 
'server.enableXsrfProtection=true'.
As a result, 'server.enableCORS' is being overridden to 'true'.

More information:
In order to protect against CSRF attacks, we send a cookie with each request.
To do so, we must specify allowable origins, which places a restriction on
cross-origin resource sharing.

If cross origin resource sharing is required, please disable server.enableXsrfProtection.
            

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.28.16.248:8501



✅ Your Streamlit app is running! Access it here: http://34.28.16.248:8501


In [63]:
import os

# The api_key variable is already in the kernel's memory from a previous step.
# We will use this existing variable to set the environment variable.
# IMPORTANT: Ensure 'api_key' contains your actual Google API Key.
os.environ['YIELD_API_KEY'] = api_key

print("YIELD_API_KEY environment variable set from existing kernel variable.")

YIELD_API_KEY environment variable set from existing kernel variable.


In [64]:
# Clear existing logs before attempting to restart
print("Clearing previous streamlit_log.txt...")
!echo '' > streamlit_log.txt
print("Previous Streamlit logs cleared.")

Clearing previous streamlit_log.txt...
Previous Streamlit logs cleared.


In [65]:
# Kill any existing Streamlit processes to ensure a clean restart
print("Attempting to kill existing Streamlit processes...")
!pkill -f streamlit || true # '|| true' prevents error if no process is found
print("Existing Streamlit processes killed (if any).")

Attempting to kill existing Streamlit processes...
^C
Existing Streamlit processes killed (if any).


In [66]:
import os
import time

print('Launching Streamlit app in background...')
# Use nohup to ensure the process continues even if the terminal is closed
# Redirect output to streamlit_log.txt for debugging if needed
!nohup streamlit run streamlit_app.py --server.port 8501 --server.enableCORS false --browser.gatherUsageStats false > streamlit_log.txt 2>&1 &

# Add a short delay for Streamlit to start and expose its URL
time.sleep(10)

print("Streamlit app launched. Checking logs for external URL...")

# Display the logs to get the external URL
try:
    with open('streamlit_log.txt', 'r') as f:
        logs = f.read()
        print(logs)
        # Extract and print the External URL from the logs
        external_url = None
        for line in logs.splitlines():
            if "External URL:" in line:
                external_url = line.split("External URL:")[1].strip()
                break
        if external_url:
            print(f"\n\u2705 Your Streamlit app is running! Access it here: {external_url}")
        else:
            print("\u274c Could not find External URL in Streamlit logs. Please check 'streamlit_log.txt' manually.")
except FileNotFoundError:
    print("Error: 'streamlit_log.txt' not found after launching Streamlit. It might not have started or logs were not written.")
except Exception as e:
    print(f"An error occurred while reading the Streamlit log file: {e}")

Launching Streamlit app in background...
Streamlit app launched. Checking logs for external URL...
2026-04-05 11:48:12.504 
'server.enableXsrfProtection=true'.
As a result, 'server.enableCORS' is being overridden to 'true'.

More information:
In order to protect against CSRF attacks, we send a cookie with each request.
To do so, we must specify allowable origins, which places a restriction on
cross-origin resource sharing.

If cross origin resource sharing is required, please disable server.enableXsrfProtection.
            

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.28.16.248:8501



✅ Your Streamlit app is running! Access it here: http://34.28.16.248:8501


In [55]:
import os

# The api_key variable is already in the kernel's memory from a previous step.
# We will use this existing variable to set the environment variable.
# IMPORTANT: Ensure 'api_key' contains your actual Google API Key.
os.environ['YIELD_API_KEY'] = api_key

print("YIELD_API_KEY environment variable set from existing kernel variable.")

YIELD_API_KEY environment variable set from existing kernel variable.


In [56]:
# Clear existing logs before attempting to restart
print("Clearing previous streamlit_log.txt...")
!echo '' > streamlit_log.txt
print("Previous Streamlit logs cleared.")

Clearing previous streamlit_log.txt...
Previous Streamlit logs cleared.


In [57]:
# Kill any existing Streamlit processes to ensure a clean restart
print("Attempting to kill existing Streamlit processes...")
!pkill -f streamlit || true # '|| true' prevents error if no process is found
print("Existing Streamlit processes killed (if any).")

Attempting to kill existing Streamlit processes...
^C
Existing Streamlit processes killed (if any).


In [58]:
import os
import time

print('Launching Streamlit app in background...')
# Use nohup to ensure the process continues even if the terminal is closed
# Redirect output to streamlit_log.txt for debugging if needed
!nohup streamlit run streamlit_app.py --server.port 8501 --server.enableCORS false --browser.gatherUsageStats false > streamlit_log.txt 2>&1 &

# Add a short delay for Streamlit to start and expose its URL
time.sleep(10)

print("Streamlit app launched. Checking logs for external URL...")

# Display the logs to get the external URL
try:
    with open('streamlit_log.txt', 'r') as f:
        logs = f.read()
        print(logs)
        # Extract and print the External URL from the logs
        external_url = None
        for line in logs.splitlines():
            if "External URL:" in line:
                external_url = line.split("External URL:")[1].strip()
                break
        if external_url:
            print(f"\n\u2705 Your Streamlit app is running! Access it here: {external_url}")
        else:
            print("\u274c Could not find External URL in Streamlit logs. Please check 'streamlit_log.txt' manually.")
except FileNotFoundError:
    print("Error: 'streamlit_log.txt' not found after launching Streamlit. It might not have started or logs were not written.")
except Exception as e:
    print(f"An error occurred while reading the Streamlit log file: {e}")

Launching Streamlit app in background...
Streamlit app launched. Checking logs for external URL...
2026-04-05 11:47:38.149 
'server.enableXsrfProtection=true'.
As a result, 'server.enableCORS' is being overridden to 'true'.

More information:
In order to protect against CSRF attacks, we send a cookie with each request.
To do so, we must specify allowable origins, which places a restriction on
cross-origin resource sharing.

If cross origin resource sharing is required, please disable server.enableXsrfProtection.
            

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.28.16.248:8501



✅ Your Streamlit app is running! Access it here: http://34.28.16.248:8501


In [51]:
import os

# The api_key variable is already in the kernel's memory from a previous step.
# We will use this existing variable to set the environment variable.
# IMPORTANT: Ensure 'api_key' contains your actual Google API Key.
os.environ['YIELD_API_KEY'] = api_key

print("YIELD_API_KEY environment variable set from existing kernel variable.")

YIELD_API_KEY environment variable set from existing kernel variable.


In [52]:
# Clear existing logs before attempting to restart
print("Clearing previous streamlit_log.txt...")
!echo '' > streamlit_log.txt
print("Previous Streamlit logs cleared.")

Clearing previous streamlit_log.txt...
Previous Streamlit logs cleared.


In [53]:
# Kill any existing Streamlit processes to ensure a clean restart
print("Attempting to kill existing Streamlit processes...")
!pkill -f streamlit || true # '|| true' prevents error if no process is found
print("Existing Streamlit processes killed (if any).")

Attempting to kill existing Streamlit processes...
^C
Existing Streamlit processes killed (if any).


In [54]:
import os
import time

print('Launching Streamlit app in background...')
# Use nohup to ensure the process continues even if the terminal is closed
# Redirect output to streamlit_log.txt for debugging if needed
!nohup streamlit run streamlit_app.py --server.port 8501 --server.enableCORS false --browser.gatherUsageStats false > streamlit_log.txt 2>&1 &

# Add a short delay for Streamlit to start and expose its URL
time.sleep(10)

print("Streamlit app launched. Checking logs for external URL...")

# Display the logs to get the external URL
try:
    with open('streamlit_log.txt', 'r') as f:
        logs = f.read()
        print(logs)
        # Extract and print the External URL from the logs
        external_url = None
        for line in logs.splitlines():
            if "External URL:" in line:
                external_url = line.split("External URL:")[1].strip()
                break
        if external_url:
            print(f"\n\u2705 Your Streamlit app is running! Access it here: {external_url}")
        else:
            print("\u274c Could not find External URL in Streamlit logs. Please check 'streamlit_log.txt' manually.")
except FileNotFoundError:
    print("Error: 'streamlit_log.txt' not found after launching Streamlit. It might not have started or logs were not written.")
except Exception as e:
    print(f"An error occurred while reading the Streamlit log file: {e}")

Launching Streamlit app in background...
Streamlit app launched. Checking logs for external URL...
2026-04-05 11:47:20.882 
'server.enableXsrfProtection=true'.
As a result, 'server.enableCORS' is being overridden to 'true'.

More information:
In order to protect against CSRF attacks, we send a cookie with each request.
To do so, we must specify allowable origins, which places a restriction on
cross-origin resource sharing.

If cross origin resource sharing is required, please disable server.enableXsrfProtection.
            

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.28.16.248:8501



✅ Your Streamlit app is running! Access it here: http://34.28.16.248:8501


In [12]:
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import os

# Assuming MODEL_PATH, yield_pipeline, hash_password, verify_password, etc.
# are defined as in your streamlit_app.py

# --- Yield Prediction Tab (extracted from streamlit_app.py) ---

# Mock load_models for demonstration purposes if running this snippet independently
# In the actual streamlit_app.py, this is handled by @st.cache_resource
def mock_load_yield_model():
    # This is a placeholder. In your full app, `yield_pipeline` is loaded properly.
    # For this snippet, we'll assume it's available or set to None.
    return (None, None, None) # Or load a dummy model for testing the UI part

yield_pipeline, _, _ = mock_load_yield_model() # Assuming the real app loads this

st.header('Yield Prediction')
with st.form('yield_form'):
    area = st.selectbox('Area', ['Zambia', 'Zimbabwe'])
    item = st.selectbox('Crop', ['Wheat', 'Maize', 'Potatoes', 'Rice, paddy', 'Sorghum', 'Soybeans'])
    year = st.number_input('Year', 2024, 2030, 2025)
    rain = st.slider('Rainfall (mm)', 0, 3500, 1000)
    pest = st.slider('Pesticides (tonnes)', 0, 150000, 5000)
    temp = st.slider('Temp (°C)', 10, 45, 25)
    if st.form_submit_button('Predict'):
        if yield_pipeline:
            input_df = pd.DataFrame([[area, item, year, rain, pest, temp]], columns=['Area', 'Item', 'Year', 'rainfall', 'pesticides', 'temperature'])
            try:
                # This line would typically use a loaded model for prediction
                # For this snippet, we'll use a placeholder for prediction output
                # pred = np.expm1(yield_pipeline.predict(input_df))[0]
                pred = 12345.67 # Placeholder prediction
                st.success(f'Estimated Yield: {pred:,.2f} hg/ha')
            except Exception as e:
                st.error(f"Error during prediction: {e}")
        else:
            st.error("Crop Yield Model not loaded. Cannot make prediction.")

2026-04-05 10:52:44.426 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 10:52:44.577 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-04-05 10:52:44.578 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 10:52:44.580 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 10:52:44.581 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 10:52:44.582 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 10:52:44.583 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 10:52:44.584 Thread 'MainThread': mi

In [6]:
%%writefile requirements.txt
streamlit
pandas
numpy
joblib
scikit-learn
xgboost
Pillow
google-generativeai
sqlalchemy
passlib
python-jose[cryptography]
tensorflow

Writing requirements.txt


In [7]:
print("Installing dependencies from requirements.txt...")
!pip install -r requirements.txt
print("Dependencies installed.")

Installing dependencies from requirements.txt...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 99.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 525.6/525.6 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.8/150.8 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 107.7 MB/s eta 0:00:00
Dependencies installed.


In [6]:
import os
from google.colab import userdata

# Retrieve the API key from Colab secrets
api_key = userdata.get('yield_key')

# Set the API key as an environment variable for the Streamlit app
os.environ['YIELD_API_KEY'] = api_key

print('Launching Streamlit app in background...')
# Use nohup to ensure the process continues even if the terminal is closed
# Redirect output to streamlit_log.txt for debugging if needed
!nohup streamlit run streamlit_app.py --server.port 8501 --server.enableCORS false --browser.gatherUsageStats false > streamlit_log.txt 2>&1 &

# Add a short delay for Streamlit to start and expose its URL
import time
time.sleep(10)

print("Streamlit app launched. Checking logs for external URL...")

# Display the logs to get the external URL
try:
    with open('streamlit_log.txt', 'r') as f:
        logs = f.read()
        print(logs)
        # Extract and print the External URL from the logs
        external_url = None
        for line in logs.splitlines():
            if "External URL:" in line:
                external_url = line.split("External URL:")[1].strip()
                break
        if external_url:
            print(f"\n\u2705 Your Streamlit app is running! Access it here: {external_url}")
        else:
            print("\u274c Could not find External URL in Streamlit logs. Please check 'streamlit_log.txt' manually.")
except FileNotFoundError:
    print("Error: 'streamlit_log.txt' not found after launching Streamlit. It might not have started or logs were not written.")
except Exception as e:
    print(f"An error occurred while reading the Streamlit log file: {e}")

Launching Streamlit app in background...
Streamlit app launched. Checking logs for external URL...
2026-04-05 11:08:53.373 
'server.enableXsrfProtection=true'.
As a result, 'server.enableCORS' is being overridden to 'true'.

More information:
In order to protect against CSRF attacks, we send a cookie with each request.
To do so, we must specify allowable origins, which places a restriction on
cross-origin resource sharing.

If cross origin resource sharing is required, please disable server.enableXsrfProtection.
            

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.28.16.248:8501



✅ Your Streamlit app is running! Access it here: http://34.28.16.248:8501


In [7]:
print("Displaying Streamlit logs from streamlit_log.txt:")
try:
    with open('streamlit_log.txt', 'r') as f:
        print(f.read())
except FileNotFoundError:
    print("Error: 'streamlit_log.txt' not found. Streamlit may not have been launched yet or logs were not written.")
except Exception as e:
    print(f"An error occurred while reading the log file: {e}")

Displaying Streamlit logs from streamlit_log.txt:
2026-04-05 11:08:53.373 
'server.enableXsrfProtection=true'.
As a result, 'server.enableCORS' is being overridden to 'true'.

More information:
In order to protect against CSRF attacks, we send a cookie with each request.
To do so, we must specify allowable origins, which places a restriction on
cross-origin resource sharing.

If cross origin resource sharing is required, please disable server.enableXsrfProtection.
            

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.28.16.248:8501




In [8]:
print('Checking for running Streamlit processes again...')
!ps aux | grep streamlit

Checking for running Streamlit processes again...
root       30884  1.3  0.5 533516 75192 ?        Sl   11:08   0:05 /usr/bin/python3 /usr/local/bin/streamlit run streamlit_app.py --server.port 8501 --server.enableCORS false --browser.gatherUsageStats false
root       32505  0.0  0.0   7372  3340 ?        S    11:15   0:00 /bin/bash -c ps aux | grep streamlit
root       32507  0.0  0.0   6480  2376 ?        S    11:15   0:00 grep streamlit


In [1]:
# Clear existing logs before attempting to restart
print("Clearing previous streamlit_log.txt and fastapi_log.txt...")
!echo '' > streamlit_log.txt
!echo '' > fastapi_log.txt
print("Previous logs cleared.")

Clearing previous streamlit_log.txt and fastapi_log.txt...
Previous logs cleared.


In [8]:
# Kill any existing uvicorn and Streamlit processes to ensure a clean restart
print("Attempting to kill existing uvicorn and Streamlit processes...")
!pkill -f uvicorn || true # '|| true' prevents error if no process is found
!pkill -f streamlit || true # '|| true' prevents error if no process is found
print("Existing uvicorn and Streamlit processes killed (if any).")

Attempting to kill existing uvicorn and Streamlit processes...
^C
^C
Existing uvicorn and Streamlit processes killed (if any).


In [3]:
import os
from google.colab import userdata

# Retrieve the API key from Colab secrets
api_key = userdata.get('yield_key')

# Set the API key as an environment variable for the Streamlit app
os.environ['YIELD_API_KEY'] = api_key

print('Launching Streamlit app in background...')
# Use nohup to ensure the process continues even if the terminal is closed
# Redirect output to streamlit_log.txt for debugging if needed
!nohup streamlit run streamlit_app.py --server.port 8501 --server.enableCORS false --browser.gatherUsageStats false > streamlit_log.txt 2>&1 &

# Add a short delay for Streamlit to start and expose its URL
import time
time.sleep(10)

print("Streamlit app launched. Checking logs for external URL...")

# Display the logs to get the external URL
try:
    with open('streamlit_log.txt', 'r') as f:
        logs = f.read()
        print(logs)
        # Extract and print the External URL from the logs
        external_url = None
        for line in logs.splitlines():
            if "External URL:" in line:
                external_url = line.split("External URL:")[1].strip()
                break
        if external_url:
            print(f"\n\u2705 Your Streamlit app is running! Access it here: {external_url}")
        else:
            print("\u274c Could not find External URL in Streamlit logs. Please check 'streamlit_log.txt' manually.")
except FileNotFoundError:
    print("Error: 'streamlit_log.txt' not found after launching Streamlit. It might not have started or logs were not written.")
except Exception as e:
    print(f"An error occurred while reading the Streamlit log file: {e}")

Launching Streamlit app in background...
Streamlit app launched. Checking logs for external URL...
Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: streamlit_app.py

❌ Could not find External URL in Streamlit logs. Please check 'streamlit_log.txt' manually.


In [ ]:
import requests
import json
import pandas as pd

# Define FastAPI URLs (ensure these match your FastAPI host/port)
API_BASE_URL = "http://127.0.0.1:8000"
REGISTER_USER_URL = f"{API_BASE_URL}/register_user"
TOKEN_URL = f"{API_BASE_URL}/token"
PREDICT_API_URL = f"{API_BASE_URL}/predict"

# --- 1. Define sample user credentials ---
TEST_USERNAME = "testuser"
TEST_PASSWORD = "testpassword123"

# --- 2. Register the test user ---
print(f"Attempting to register user: {TEST_USERNAME}")
try:
    register_payload = {"username": TEST_USERNAME, "hashed_password": TEST_PASSWORD}
    register_response = requests.post(REGISTER_USER_URL, json=register_payload)
    register_response.raise_for_status() # Raise an exception for HTTP errors
    print("User registered successfully (or already exists).")
except requests.exceptions.HTTPError as e:
    if e.response.status_code == 400 and "Username already registered" in e.response.text:
        print("User already registered.")
    else:
        print(f"Error registering user: {e.response.status_code} - {e.response.text}")
        exit()
except requests.exceptions.ConnectionError:
    print(f"Error: Could not connect to FastAPI at {API_BASE_URL}. Is the server running?")
    exit()

# --- 3. Log in to get an access token ---
print(f"Attempting to log in user: {TEST_USERNAME}")
try:
    token_payload = {"username": TEST_USERNAME, "password": TEST_PASSWORD}
    token_response = requests.post(TOKEN_URL, data=token_payload)
    token_response.raise_for_status() # Raise an exception for HTTP errors
    token_data = token_response.json()
    access_token = token_data['access_token']
    token_type = token_data['token_type']
    print("Successfully obtained access token.")
except requests.exceptions.HTTPError as e:
    print(f"Error logging in: {e.response.status_code} - {e.response.text}")
    exit()

# --- 4. Prepare headers with the access token ---
headers = {
    "Authorization": f"{token_type} {access_token}",
    "Content-Type": "application/json"
}

# --- 5. Use existing sample_data for prediction (assuming it's in the kernel) ---
# If sample_data is not available, uncomment and use the following:
# sample_data = pd.DataFrame({
#     'Area': ['India', 'Brazil', 'France', 'India'],
#     'Item': ['Wheat', 'Maize', 'Potatoes', 'Rice, paddy'],
#     'Year': [2022, 2023, 2022, 2021],
#     'rainfall': [1200.0, 1800.0, 700.0, 1100.0],
#     'pesticides': [60000.0, 80000.0, 30000.0, 55000.0],
#     'temperature': [25.5, 28.0, 15.0, 26.5]
# })

if 'sample_data' not in globals():
    print("Error: 'sample_data' DataFrame not found in current session. Please ensure the cell defining it has been run.")
    exit()

# Convert sample_data DataFrame to a list of dictionaries for JSON payload
predict_payload = sample_data.to_dict(orient='records')

# --- 6. Make the prediction request ---
print("\nMaking prediction request...")
try:
    predict_response = requests.post(PREDICT_API_URL, headers=headers, data=json.dumps(predict_payload))
    predict_response.raise_for_status() # Raise an exception for HTTP errors
    predictions = predict_response.json()

    print("Prediction successful!")
    print("Predicted Crop Yields (hg/ha):")
    for i, pred in enumerate(predictions['predictions']):
        print(f"  Input {i+1}: {pred:,.0f} hg/ha")

except requests.exceptions.HTTPError as e:
    print(f"Error during prediction: {e.response.status_code} - {e.response.text}")
except requests.exceptions.ConnectionError:
    print(f"Error: Could not connect to FastAPI at {API_BASE_URL}. Is the server running?")
except Exception as e:
    print(f"An unexpected error occurred during prediction: {e}")

In [4]:
print("Installing sqlalchemy...")
!pip install sqlalchemy
print("sqlalchemy installed.")

Installing sqlalchemy...
sqlalchemy installed.


In [2]:
# Clear existing logs before attempting to restart
print("Clearing previous streamlit_log.txt and fastapi_log.txt...")
!echo '' > streamlit_log.txt
!echo '' > fastapi_log.txt
print("Previous logs cleared.")

Clearing previous streamlit_log.txt and fastapi_log.txt...
Previous logs cleared.


In [3]:
# Kill any existing uvicorn and Streamlit processes to ensure a clean restart
print("Attempting to kill existing uvicorn and Streamlit processes...")
!pkill -f uvicorn || true # '|| true' prevents error if no process is found
!pkill -f streamlit || true # '|| true' prevents error if no process is found
print("Existing uvicorn and Streamlit processes killed (if any).")

Attempting to kill existing uvicorn and Streamlit processes...
^C
^C
Existing uvicorn and Streamlit processes killed (if any).


In [ ]:
# Run the FastAPI app in the background
# The '&' at the end sends the process to the background
# The 'nohup' command ensures the process continues even if the terminal is closed
print('Launching FastAPI app in background...')
!nohup uvicorn main:app --host 0.0.0.0 --port 8000 > fastapi_log.txt 2>&1 &

# Add a short delay to allow the server to start before the notebook continues
import time
time.sleep(5)

print("FastAPI app launched. Check 'fastapi_log.txt' for logs if needed.")

In [12]:
import os
# from google.colab import userdata # Removed as it causes TimeoutException in background process

# The api_key variable is already in the kernel's memory from a previous step.
# We will use this existing variable to set the environment variable.
# IMPORTANT: Ensure 'api_key' contains your actual Google API Key.
# api_key = userdata.get('yield_key') # Removed this line

# Assuming 'api_key' is already defined in the kernel from a prior successful execution
# If api_key is not defined, this cell will fail.

os.environ['YIELD_API_KEY'] = api_key

print('Launching Streamlit app in background...')
# Use nohup to ensure the process continues even if the terminal is closed
# Redirect output to streamlit_log.txt for debugging if needed
!nohup streamlit run streamlit_app.py --server.port 8501 --server.enableCORS false --browser.gatherUsageStats false > streamlit_log.txt 2>&1 &

# Add a short delay for Streamlit to start and expose its URL
import time
time.sleep(10)

print("Streamlit app launched. Checking logs for external URL...")

# Display the logs to get the external URL
try:
    with open('streamlit_log.txt', 'r') as f:
        logs = f.read()
        print(logs)
        # Extract and print the External URL from the logs
        external_url = None
        for line in logs.splitlines():
            if "External URL:" in line:
                external_url = line.split("External URL:")[1].strip()
                break
        if external_url:
            print(f"\n\u2705 Your Streamlit app is running! Access it here: {external_url}")
        else:
            print("\u274c Could not find External URL in Streamlit logs. Please check 'streamlit_log.txt' manually.")
except FileNotFoundError:
    print("Error: 'streamlit_log.txt' not found after launching Streamlit. It might not have started or logs were not written.")
except Exception as e:
    print(f"An error occurred while reading the Streamlit log file: {e}")

Launching Streamlit app in background...
Streamlit app launched. Checking logs for external URL...
2026-04-05 15:31:14.343 
'server.enableXsrfProtection=true'.
As a result, 'server.enableCORS' is being overridden to 'true'.

More information:
In order to protect against CSRF attacks, we send a cookie with each request.
To do so, we must specify allowable origins, which places a restriction on
cross-origin resource sharing.

If cross origin resource sharing is required, please disable server.enableXsrfProtection.
            

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.11.40.11:8501



✅ Your Streamlit app is running! Access it here: http://34.11.40.11:8501


In [7]:
print("Attempting to kill existing uvicorn processes...")
!pkill -f uvicorn || true # '|| true' prevents error if no process is found
print("Existing uvicorn processes killed (if any).")

Attempting to kill existing uvicorn processes...
^C
Existing uvicorn processes killed (if any).


In [11]:
print('Listing current directory contents:')
!ls -F


Listing current directory contents:
drive/		 market_trends.csv  sample_data/       yield_df.csv
fastapi_log.txt  model_artifacts/   streamlit_app.py
main.py		 requirements.txt   streamlit_log.txt


In [12]:
print('Attempting to kill existing uvicorn processes...')
!pkill -f uvicorn || true # '|| true' prevents error if no process is found
print("Existing uvicorn processes killed (if any).")


Attempting to kill existing uvicorn processes...
^C
Existing uvicorn processes killed (if any).


In [13]:
print('Checking for running FastAPI process (uvicorn) again...')
!ps aux | grep uvicorn

Checking for running FastAPI process (uvicorn) again...
root       33491  0.0  0.0   7372  3504 ?        S    11:19   0:00 /bin/bash -c ps aux | grep uvicorn
root       33493  0.0  0.0   6480  2376 ?        S    11:19   0:00 grep uvicorn


In [18]:
# Clear existing Streamlit logs
print("Clearing previous streamlit_log.txt...")
!echo '' > streamlit_log.txt
print("Previous Streamlit logs cleared.")

Clearing previous streamlit_log.txt...
Previous Streamlit logs cleared.


In [19]:
# Kill any existing Streamlit processes to ensure a clean restart
print("Attempting to kill existing Streamlit processes...")
!pkill -f streamlit || true # '|| true' prevents error if no process is found
print("Existing Streamlit processes killed (if any).")

Attempting to kill existing Streamlit processes...
^C
Existing Streamlit processes killed (if any).


In [20]:
import os
from google.colab import userdata

# Retrieve the API key from Colab secrets
api_key = userdata.get('yield_key')

# Set the API key as an environment variable for the Streamlit app
os.environ['YIELD_API_KEY'] = api_key

print('Launching Streamlit app in background...')
# Use nohup to ensure the process continues even if the terminal is closed
# Redirect output to streamlit_log.txt for debugging if needed
!nohup streamlit run streamlit_app.py --server.port 8501 --server.enableCORS false --browser.gatherUsageStats false > streamlit_log.txt 2>&1 &

# Add a short delay for Streamlit to start and expose its URL
import time
time.sleep(10)

print("Streamlit app launched. Checking logs for external URL...")

# Display the logs to get the external URL
try:
    with open('streamlit_log.txt', 'r') as f:
        logs = f.read()
        print(logs)
        # Extract and print the External URL from the logs
        external_url = None
        for line in logs.splitlines():
            if "External URL:" in line:
                external_url = line.split("External URL:")[1].strip()
                break
        if external_url:
            print(f"\n\u2705 Your Streamlit app is running! Access it here: {external_url}")
        else:
            print("\u274c Could not find External URL in Streamlit logs. Please check 'streamlit_log.txt' manually.")
except FileNotFoundError:
    print("Error: 'streamlit_log.txt' not found after launching Streamlit. It might not have started or logs were not written.")
except Exception as e:
    print(f"An error occurred while reading the Streamlit log file: {e}")

TimeoutException: Requesting secret yield_key timed out. Secrets can only be fetched when running from the Colab UI.

In [21]:
print('Checking for running FastAPI process (uvicorn) again...')
!ps aux | grep uvicorn

Checking for running FastAPI process (uvicorn) again...
root       34320  0.0  0.0   7372  3524 ?        S    11:22   0:00 /bin/bash -c ps aux | grep uvicorn
root       34322  0.0  0.0   6480  2428 ?        S    11:22   0:00 grep uvicorn


In [59]:
import os

# The api_key variable is already in the kernel's memory from a previous step.
# We will use this existing variable to set the environment variable.
# IMPORTANT: Ensure 'api_key' contains your actual Google API Key.
os.environ['YIELD_API_KEY'] = api_key

print("YIELD_API_KEY environment variable set from existing kernel variable.")

YIELD_API_KEY environment variable set from existing kernel variable.


In [60]:
# Clear existing logs before attempting to restart
print("Clearing previous streamlit_log.txt...")
!echo '' > streamlit_log.txt
print("Previous Streamlit logs cleared.")

Clearing previous streamlit_log.txt...
Previous Streamlit logs cleared.


In [61]:
# Kill any existing Streamlit processes to ensure a clean restart
print("Attempting to kill existing Streamlit processes...")
!pkill -f streamlit || true # '|| true' prevents error if no process is found
print("Existing Streamlit processes killed (if any).")

Attempting to kill existing Streamlit processes...
^C
Existing Streamlit processes killed (if any).


In [62]:
import os
import time

print('Launching Streamlit app in background...')
# Use nohup to ensure the process continues even if the terminal is closed
# Redirect output to streamlit_log.txt for debugging if needed
!nohup streamlit run streamlit_app.py --server.port 8501 --server.enableCORS false --browser.gatherUsageStats false > streamlit_log.txt 2>&1 &

# Add a short delay for Streamlit to start and expose its URL
time.sleep(10)

print("Streamlit app launched. Checking logs for external URL...")

# Display the logs to get the external URL
try:
    with open('streamlit_log.txt', 'r') as f:
        logs = f.read()
        print(logs)
        # Extract and print the External URL from the logs
        external_url = None
        for line in logs.splitlines():
            if "External URL:" in line:
                external_url = line.split("External URL:")[1].strip()
                break
        if external_url:
            print(f"\n\u2705 Your Streamlit app is running! Access it here: {external_url}")
        else:
            print("\u274c Could not find External URL in Streamlit logs. Please check 'streamlit_log.txt' manually.")
except FileNotFoundError:
    print("Error: 'streamlit_log.txt' not found after launching Streamlit. It might not have started or logs were not written.")
except Exception as e:
    print(f"An error occurred while reading the Streamlit log file: {e}")

Launching Streamlit app in background...
Streamlit app launched. Checking logs for external URL...
2026-04-05 11:47:52.012 
'server.enableXsrfProtection=true'.
As a result, 'server.enableCORS' is being overridden to 'true'.

More information:
In order to protect against CSRF attacks, we send a cookie with each request.
To do so, we must specify allowable origins, which places a restriction on
cross-origin resource sharing.

If cross origin resource sharing is required, please disable server.enableXsrfProtection.
            

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.28.16.248:8501



✅ Your Streamlit app is running! Access it here: http://34.28.16.248:8501


In [46]:
import os

# The api_key variable is already in the kernel's memory from a previous step.
# We will use this existing variable to set the environment variable.
# IMPORTANT: Ensure 'api_key' contains your actual Google API Key.
os.environ['YIELD_API_KEY'] = api_key

print("YIELD_API_KEY environment variable set from existing kernel variable.")

YIELD_API_KEY environment variable set from existing kernel variable.


In [47]:
# Clear existing logs before attempting to restart
print("Clearing previous streamlit_log.txt...")
!echo '' > streamlit_log.txt
print("Previous Streamlit logs cleared.")

Clearing previous streamlit_log.txt...
Previous Streamlit logs cleared.


In [48]:
# Kill any existing Streamlit processes to ensure a clean restart
print("Attempting to kill existing Streamlit processes...")
!pkill -f streamlit || true # '|| true' prevents error if no process is found
print("Existing Streamlit processes killed (if any).")

Attempting to kill existing Streamlit processes...
^C
Existing Streamlit processes killed (if any).


In [49]:
import os
import time

print('Launching Streamlit app in background...')
# Use nohup to ensure the process continues even if the terminal is closed
# Redirect output to streamlit_log.txt for debugging if needed
!nohup streamlit run streamlit_app.py --server.port 8501 --server.enableCORS false --browser.gatherUsageStats false > streamlit_log.txt 2>&1 &

# Add a short delay for Streamlit to start and expose its URL
time.sleep(10)

print("Streamlit app launched. Checking logs for external URL...")

# Display the logs to get the external URL
try:
    with open('streamlit_log.txt', 'r') as f:
        logs = f.read()
        print(logs)
        # Extract and print the External URL from the logs
        external_url = None
        for line in logs.splitlines():
            if "External URL:" in line:
                external_url = line.split("External URL:")[1].strip()
                break
        if external_url:
            print(f"\n\u2705 Your Streamlit app is running! Access it here: {external_url}")
        else:
            print("\u274c Could not find External URL in Streamlit logs. Please check 'streamlit_log.txt' manually.")
except FileNotFoundError:
    print("Error: 'streamlit_log.txt' not found after launching Streamlit. It might not have started or logs were not written.")
except Exception as e:
    print(f"An error occurred while reading the Streamlit log file: {e}")

Launching Streamlit app in background...
Streamlit app launched. Checking logs for external URL...
2026-04-05 11:47:01.098 
'server.enableXsrfProtection=true'.
As a result, 'server.enableCORS' is being overridden to 'true'.

More information:
In order to protect against CSRF attacks, we send a cookie with each request.
To do so, we must specify allowable origins, which places a restriction on
cross-origin resource sharing.

If cross origin resource sharing is required, please disable server.enableXsrfProtection.
            

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.28.16.248:8501



✅ Your Streamlit app is running! Access it here: http://34.28.16.248:8501


In [42]:
import os

# The api_key variable is already in the kernel's memory from a previous step.
# We will use this existing variable to set the environment variable.
# IMPORTANT: Ensure 'api_key' contains your actual Google API Key.
os.environ['YIELD_API_KEY'] = api_key

print("YIELD_API_KEY environment variable set from existing kernel variable.")

YIELD_API_KEY environment variable set from existing kernel variable.


In [43]:
# Clear existing logs before attempting to restart
print("Clearing previous streamlit_log.txt...")
!echo '' > streamlit_log.txt
print("Previous Streamlit logs cleared.")

Clearing previous streamlit_log.txt...
Previous Streamlit logs cleared.


In [44]:
# Kill any existing Streamlit processes to ensure a clean restart
print("Attempting to kill existing Streamlit processes...")
!pkill -f streamlit || true # '|| true' prevents error if no process is found
print("Existing Streamlit processes killed (if any).")

Attempting to kill existing Streamlit processes...
^C
Existing Streamlit processes killed (if any).


In [45]:
import os
import time

print('Launching Streamlit app in background...')
# Use nohup to ensure the process continues even if the terminal is closed
# Redirect output to streamlit_log.txt for debugging if needed
!nohup streamlit run streamlit_app.py --server.port 8501 --server.enableCORS false --browser.gatherUsageStats false > streamlit_log.txt 2>&1 &

# Add a short delay for Streamlit to start and expose its URL
time.sleep(10)

print("Streamlit app launched. Checking logs for external URL...")

# Display the logs to get the external URL
try:
    with open('streamlit_log.txt', 'r') as f:
        logs = f.read()
        print(logs)
        # Extract and print the External URL from the logs
        external_url = None
        for line in logs.splitlines():
            if "External URL:" in line:
                external_url = line.split("External URL:")[1].strip()
                break
        if external_url:
            print(f"\n\u2705 Your Streamlit app is running! Access it here: {external_url}")
        else:
            print("\u274c Could not find External URL in Streamlit logs. Please check 'streamlit_log.txt' manually.")
except FileNotFoundError:
    print("Error: 'streamlit_log.txt' not found after launching Streamlit. It might not have started or logs were not written.")
except Exception as e:
    print(f"An error occurred while reading the Streamlit log file: {e}")

Launching Streamlit app in background...
Streamlit app launched. Checking logs for external URL...
2026-04-05 11:42:09.984 
'server.enableXsrfProtection=true'.
As a result, 'server.enableCORS' is being overridden to 'true'.

More information:
In order to protect against CSRF attacks, we send a cookie with each request.
To do so, we must specify allowable origins, which places a restriction on
cross-origin resource sharing.

If cross origin resource sharing is required, please disable server.enableXsrfProtection.
            

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.28.16.248:8501



✅ Your Streamlit app is running! Access it here: http://34.28.16.248:8501


In [38]:
import os
from google.colab import userdata

# IMPORTANT: Make sure you have your Gemini API key stored as a secret named 'yield_key' in Colab.
# You can add a new secret by clicking on the '🔑' icon in the left sidebar.
# Learn more: https://colab.research.google.com/notebooks/snippets/secrets.ipynb
api_key = userdata.get('yield_key')

# Set the API key as an environment variable
os.environ['YIELD_API_KEY'] = api_key

print("YIELD_API_KEY environment variable set.")

YIELD_API_KEY environment variable set.


In [39]:
# Clear existing logs before attempting to restart
print("Clearing previous streamlit_log.txt...")
!echo '' > streamlit_log.txt
print("Previous Streamlit logs cleared.")

Clearing previous streamlit_log.txt...
Previous Streamlit logs cleared.


In [40]:
# Kill any existing Streamlit processes to ensure a clean restart
print("Attempting to kill existing Streamlit processes...")
!pkill -f streamlit || true # '|| true' prevents error if no process is found
print("Existing Streamlit processes killed (if any).")

Attempting to kill existing Streamlit processes...
^C
Existing Streamlit processes killed (if any).


In [41]:
import os
import time

print('Launching Streamlit app in background...')
# Use nohup to ensure the process continues even if the terminal is closed
# Redirect output to streamlit_log.txt for debugging if needed
!nohup streamlit run streamlit_app.py --server.port 8501 --server.enableCORS false --browser.gatherUsageStats false > streamlit_log.txt 2>&1 &

# Add a short delay for Streamlit to start and expose its URL
time.sleep(10)

print("Streamlit app launched. Checking logs for external URL...")

# Display the logs to get the external URL
try:
    with open('streamlit_log.txt', 'r') as f:
        logs = f.read()
        print(logs)
        # Extract and print the External URL from the logs
        external_url = None
        for line in logs.splitlines():
            if "External URL:" in line:
                external_url = line.split("External URL:")[1].strip()
                break
        if external_url:
            print(f"\n\u2705 Your Streamlit app is running! Access it here: {external_url}")
        else:
            print("\u274c Could not find External URL in Streamlit logs. Please check 'streamlit_log.txt' manually.")
except FileNotFoundError:
    print("Error: 'streamlit_log.txt' not found after launching Streamlit. It might not have started or logs were not written.")
except Exception as e:
    print(f"An error occurred while reading the Streamlit log file: {e}")

Launching Streamlit app in background...
Streamlit app launched. Checking logs for external URL...
2026-04-05 11:41:53.921 
'server.enableXsrfProtection=true'.
As a result, 'server.enableCORS' is being overridden to 'true'.

More information:
In order to protect against CSRF attacks, we send a cookie with each request.
To do so, we must specify allowable origins, which places a restriction on
cross-origin resource sharing.

If cross origin resource sharing is required, please disable server.enableXsrfProtection.
            

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.28.16.248:8501



✅ Your Streamlit app is running! Access it here: http://34.28.16.248:8501


In [34]:
import os
from google.colab import userdata

# IMPORTANT: Make sure you have your Gemini API key stored as a secret named 'yield_key' in Colab.
# You can add a new secret by clicking on the '🔑' icon in the left sidebar.
# Learn more: https://colab.research.google.com/notebooks/snippets/secrets.ipynb
api_key = userdata.get('yield_key')

# Set the API key as an environment variable
os.environ['YIELD_API_KEY'] = api_key

print("YIELD_API_KEY environment variable set.")

YIELD_API_KEY environment variable set.


In [35]:
# Clear existing logs before attempting to restart
print("Clearing previous streamlit_log.txt...")
!echo '' > streamlit_log.txt
print("Previous Streamlit logs cleared.")

Clearing previous streamlit_log.txt...
Previous Streamlit logs cleared.


In [36]:
# Kill any existing Streamlit processes to ensure a clean restart
print("Attempting to kill existing Streamlit processes...")
!pkill -f streamlit || true # '|| true' prevents error if no process is found
print("Existing Streamlit processes killed (if any).")

Attempting to kill existing Streamlit processes...
^C
Existing Streamlit processes killed (if any).


In [37]:
import os
import time

print('Launching Streamlit app in background...')
# Use nohup to ensure the process continues even if the terminal is closed
# Redirect output to streamlit_log.txt for debugging if needed
!nohup streamlit run streamlit_app.py --server.port 8501 --server.enableCORS false --browser.gatherUsageStats false > streamlit_log.txt 2>&1 &

# Add a short delay for Streamlit to start and expose its URL
time.sleep(10)

print("Streamlit app launched. Checking logs for external URL...")

# Display the logs to get the external URL
try:
    with open('streamlit_log.txt', 'r') as f:
        logs = f.read()
        print(logs)
        # Extract and print the External URL from the logs
        external_url = None
        for line in logs.splitlines():
            if "External URL:" in line:
                external_url = line.split("External URL:")[1].strip()
                break
        if external_url:
            print(f"\n\u2705 Your Streamlit app is running! Access it here: {external_url}")
        else:
            print("\u274c Could not find External URL in Streamlit logs. Please check 'streamlit_log.txt' manually.")
except FileNotFoundError:
    print("Error: 'streamlit_log.txt' not found after launching Streamlit. It might not have started or logs were not written.")
except Exception as e:
    print(f"An error occurred while reading the Streamlit log file: {e}")

Launching Streamlit app in background...
Streamlit app launched. Checking logs for external URL...
2026-04-05 11:41:37.727 
'server.enableXsrfProtection=true'.
As a result, 'server.enableCORS' is being overridden to 'true'.

More information:
In order to protect against CSRF attacks, we send a cookie with each request.
To do so, we must specify allowable origins, which places a restriction on
cross-origin resource sharing.

If cross origin resource sharing is required, please disable server.enableXsrfProtection.
            

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.28.16.248:8501



✅ Your Streamlit app is running! Access it here: http://34.28.16.248:8501


In [33]:
import os
from google.colab import userdata

# IMPORTANT: Make sure you have your Gemini API key stored as a secret named 'yield_key' in Colab.
# You can add a new secret by clicking on the '🔑' icon in the left sidebar.
# Learn more: https://colab.research.google.com/notebooks/snippets/secrets.ipynb
api_key = userdata.get('yield_key')

# Set the API key as an environment variable
os.environ['YIELD_API_KEY'] = api_key

print("YIELD_API_KEY environment variable set.")

YIELD_API_KEY environment variable set.


In [32]:
import os
from google.colab import userdata

# IMPORTANT: Make sure you have your Gemini API key stored as a secret named 'yield_key' in Colab.
# You can add a new secret by clicking on the '🔑' icon in the left sidebar.
# Learn more: https://colab.research.google.com/notebooks/snippets/secrets.ipynb
api_key = userdata.get('yield_key')

# Set the API key as an environment variable
os.environ['YIELD_API_KEY'] = api_key

print("YIELD_API_KEY environment variable set.")

YIELD_API_KEY environment variable set.


In [28]:
import os
from google.colab import userdata

# IMPORTANT: Make sure you have your Gemini API key stored as a secret named 'yield_key' in Colab.
# You can add a new secret by clicking on the '🔑' icon in the left sidebar.
# Learn more: https://colab.research.google.com/notebooks/snippets/secrets.ipynb
api_key = userdata.get('yield_key')

# Set the API key as an environment variable
os.environ['YIELD_API_KEY'] = api_key

print("YIELD_API_KEY environment variable set.")

YIELD_API_KEY environment variable set.


In [29]:
# Clear existing logs before attempting to restart
print("Clearing previous streamlit_log.txt...")
!echo '' > streamlit_log.txt
print("Previous Streamlit logs cleared.")

Clearing previous streamlit_log.txt...
Previous Streamlit logs cleared.


In [30]:
# Kill any existing Streamlit processes to ensure a clean restart
print("Attempting to kill existing Streamlit processes...")
!pkill -f streamlit || true # '|| true' prevents error if no process is found
print("Existing Streamlit processes killed (if any).")

Attempting to kill existing Streamlit processes...
^C
Existing Streamlit processes killed (if any).


In [31]:
import os
import time

print('Launching Streamlit app in background...')
# Use nohup to ensure the process continues even if the terminal is closed
# Redirect output to streamlit_log.txt for debugging if needed
!nohup streamlit run streamlit_app.py --server.port 8501 --server.enableCORS false --browser.gatherUsageStats false > streamlit_log.txt 2>&1 &

# Add a short delay for Streamlit to start and expose its URL
time.sleep(10)

print("Streamlit app launched. Checking logs for external URL...")

# Display the logs to get the external URL
try:
    with open('streamlit_log.txt', 'r') as f:
        logs = f.read()
        print(logs)
        # Extract and print the External URL from the logs
        external_url = None
        for line in logs.splitlines():
            if "External URL:" in line:
                external_url = line.split("External URL:")[1].strip()
                break
        if external_url:
            print(f"\n\u2705 Your Streamlit app is running! Access it here: {external_url}")
        else:
            print("\u274c Could not find External URL in Streamlit logs. Please check 'streamlit_log.txt' manually.")
except FileNotFoundError:
    print("Error: 'streamlit_log.txt' not found after launching Streamlit. It might not have started or logs were not written.")
except Exception as e:
    print(f"An error occurred while reading the Streamlit log file: {e}")

Launching Streamlit app in background...
Streamlit app launched. Checking logs for external URL...
2026-04-05 11:41:19.024 
'server.enableXsrfProtection=true'.
As a result, 'server.enableCORS' is being overridden to 'true'.

More information:
In order to protect against CSRF attacks, we send a cookie with each request.
To do so, we must specify allowable origins, which places a restriction on
cross-origin resource sharing.

If cross origin resource sharing is required, please disable server.enableXsrfProtection.
            

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.28.16.248:8501



✅ Your Streamlit app is running! Access it here: http://34.28.16.248:8501


In [24]:
import os
from google.colab import userdata

# IMPORTANT: Make sure you have your Gemini API key stored as a secret named 'yield_key' in Colab.
# You can add a new secret by clicking on the '🔑' icon in the left sidebar.
# Learn more: https://colab.research.google.com/notebooks/snippets/secrets.ipynb
api_key = userdata.get('yield_key')

# Set the API key as an environment variable
os.environ['YIELD_API_KEY'] = api_key

print("YIELD_API_KEY environment variable set.")

YIELD_API_KEY environment variable set.


In [25]:
# Clear existing logs before attempting to restart
print("Clearing previous streamlit_log.txt...")
!echo '' > streamlit_log.txt
print("Previous Streamlit logs cleared.")

Clearing previous streamlit_log.txt...
Previous Streamlit logs cleared.


In [26]:
# Kill any existing Streamlit processes to ensure a clean restart
print("Attempting to kill existing Streamlit processes...")
!pkill -f streamlit || true # '|| true' prevents error if no process is found
print("Existing Streamlit processes killed (if any).")

Attempting to kill existing Streamlit processes...
^C
Existing Streamlit processes killed (if any).


In [27]:
import os
import time

print('Launching Streamlit app in background...')
# Use nohup to ensure the process continues even if the terminal is closed
# Redirect output to streamlit_log.txt for debugging if needed
!nohup streamlit run streamlit_app.py --server.port 8501 --server.enableCORS false --browser.gatherUsageStats false > streamlit_log.txt 2>&1 &

# Add a short delay for Streamlit to start and expose its URL
time.sleep(10)

print("Streamlit app launched. Checking logs for external URL...")

# Display the logs to get the external URL
try:
    with open('streamlit_log.txt', 'r') as f:
        logs = f.read()
        print(logs)
        # Extract and print the External URL from the logs
        external_url = None
        for line in logs.splitlines():
            if "External URL:" in line:
                external_url = line.split("External URL:")[1].strip()
                break
        if external_url:
            print(f"\n\u2705 Your Streamlit app is running! Access it here: {external_url}")
        else:
            print("\u274c Could not find External URL in Streamlit logs. Please check 'streamlit_log.txt' manually.")
except FileNotFoundError:
    print("Error: 'streamlit_log.txt' not found after launching Streamlit. It might not have started or logs were not written.")
except Exception as e:
    print(f"An error occurred while reading the Streamlit log file: {e}")

Launching Streamlit app in background...
Streamlit app launched. Checking logs for external URL...
2026-04-05 11:41:04.368 
'server.enableXsrfProtection=true'.
As a result, 'server.enableCORS' is being overridden to 'true'.

More information:
In order to protect against CSRF attacks, we send a cookie with each request.
To do so, we must specify allowable origins, which places a restriction on
cross-origin resource sharing.

If cross origin resource sharing is required, please disable server.enableXsrfProtection.
            

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.28.16.248:8501



✅ Your Streamlit app is running! Access it here: http://34.28.16.248:8501


In [10]:
# Clear existing logs before attempting to restart
print("Clearing previous streamlit_log.txt and fastapi_log.txt...")
!echo '' > streamlit_log.txt
!echo '' > fastapi_log.txt
print("Previous logs cleared.")

Clearing previous streamlit_log.txt and fastapi_log.txt...
Previous logs cleared.


In [11]:
# Kill any existing uvicorn and Streamlit processes to ensure a clean restart
print("Attempting to kill existing uvicorn and Streamlit processes...")
!pkill -f uvicorn || true # '|| true' prevents error if no process is found
!pkill -f streamlit || true # '|| true' prevents error if no process is found
print("Existing uvicorn and Streamlit processes killed (if any).")

Attempting to kill existing uvicorn and Streamlit processes...
^C
^C
Existing uvicorn and Streamlit processes killed (if any).


In [16]:
# Run the FastAPI app in the background
# The '&' at the end sends the process to the background
# The 'nohup' command ensures the process continues even if the terminal is closed
print('Launching FastAPI app in background...')
!nohup uvicorn main:app --host 0.0.0.0 --port 8000 > fastapi_log.txt 2>&1 &

# Add a short delay to allow the server to start before the notebook continues
import time
time.sleep(5)

print("FastAPI app launched. Check 'fastapi_log.txt' for logs if needed.")

Launching FastAPI app in background...
FastAPI app launched. Check 'fastapi_log.txt' for logs if needed.


In [17]:
import os
from google.colab import userdata

# Retrieve the API key from Colab secrets
api_key = userdata.get('yield_key')

# Set the API key as an environment variable for the Streamlit app
os.environ['YIELD_API_KEY'] = api_key

print('Launching Streamlit app in background...')
# Use nohup to ensure the process continues even if the terminal is closed
# Redirect output to streamlit_log.txt for debugging if needed
!nohup streamlit run streamlit_app.py --server.port 8501 --server.enableCORS false --browser.gatherUsageStats false > streamlit_log.txt 2>&1 &

# Add a short delay for Streamlit to start and expose its URL
import time
time.sleep(10)

print("Streamlit app launched. Checking logs for external URL...")

# Display the logs to get the external URL
try:
    with open('streamlit_log.txt', 'r') as f:
        logs = f.read()
        print(logs)
        # Extract and print the External URL from the logs
        external_url = None
        for line in logs.splitlines():
            if "External URL:" in line:
                external_url = line.split("External URL:")[1].strip()
                break
        if external_url:
            print(f"\n\u2705 Your Streamlit app is running! Access it here: {external_url}")
        else:
            print("\u274c Could not find External URL in Streamlit logs. Please check 'streamlit_log.txt' manually.")
except FileNotFoundError:
    print("Error: 'streamlit_log.txt' not found after launching Streamlit. It might not have started or logs were not written.")
except Exception as e:
    print(f"An error occurred while reading the Streamlit log file: {e}")

TimeoutException: Requesting secret yield_key timed out. Secrets can only be fetched when running from the Colab UI.

In [8]:
print('Checking for running FastAPI process (uvicorn) again...')
!ps aux | grep uvicorn

Checking for running FastAPI process (uvicorn) again...
root       25935  0.0  0.0   7372  3412 ?        S    10:48   0:00 /bin/bash -c ps aux | grep uvicorn
root       25937  0.0  0.0   6480  2528 ?        S    10:48   0:00 grep uvicorn


In [9]:
try:
    import sqlalchemy
    print(f"SQLAlchemy is installed. Version: {sqlalchemy.__version__}")
except ImportError:
    print("SQLAlchemy is NOT installed.")

SQLAlchemy is installed. Version: 2.0.48
